In [1]:
!ls

'0 - NgScraperPol.ipynb'
'1 - Data preparation.ipynb'
'2 - Reliability - 5Fold - Separate.ipynb'
'3 - Reliability - 5Fold - Separate - mistral22B.ipynb'
'4 - Plotter.ipynb'
'5 - Political Error Analysis.ipynb'
'6 - Domain inpact analisys.ipynb'
 Dataset
 Results


In [2]:
import pandas as pd

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib

# Use the PGF backend
matplotlib.use("pgf")

# Configure LaTeX rendering
plt.rcParams.update({
    "pgf.texsystem": "pdflatex",  # or "lualatex", "xelatex"
    "text.usetex": True,
    "font.family": "serif",  # Use LaTeX default serif font
    "font.serif": [],        # Let LaTeX choose the font
    "axes.labelsize": 10,    # LaTeX font sizes
    "font.size": 10,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "pgf.rcfonts": False,    # Do not override LaTeX settings
})

# Adding political inclination to results dataframe using NewsGuard data

In [4]:
ng = pd.read_csv('Dataset/newsguard.csv')

df = pd.read_csv('Dataset/CleannedDataset/newDataframeWithOldCleanedPoliticsVersion.csv')

In [5]:
# --- 1. Keep only what we need from ng and align column names ----------
ng_subset = (
    ng[['Domain', 'Orientation']]                      # Keep the two relevant columns
      .drop_duplicates(subset='Domain')                # Ensure no duplicate domains
      .rename(columns={'Domain': 'domain'})            # Rename so it matches df
)

# --- 2. Left-join df with ng on the domain -----------------------------
patchedDf = (
    df.merge(
        ng_subset,
        on='domain',
        how='left',
        indicator=False                                # We’ll generate the “no match” list later
    )
)

# --- 3. Fill missing Orientation values with the placeholder ----------
patchedDf['Orientation'] = patchedDf['Orientation'].fillna('Non labelled')

# --- 4. Extract the rows that had no match (i.e. got the placeholder) --
unmatched_rows = patchedDf[patchedDf['Orientation'] == 'Non labelled']

# --- 5. (Optional) show or save the results ---------------------------
print(unmatched_rows)                                  # Rows whose domains weren’t found in ng


                                                     url  \
98            https://middlesboronews.com/category/news/   
99     https://middlesboronews.com/2025/03/19/zachery...   
100    https://middlesboronews.com/2025/03/19/bobcats...   
101    https://middlesboronews.com/2025/03/18/saylor-...   
102    https://middlesboronews.com/2025/03/17/lady-li...   
...                                                  ...   
19111  https://www.politicshome.com/news/article/furl...   
19112  https://www.politicshome.com/news/article/stat...   
19113  https://www.politicshome.com/news/article/brex...   
19114  https://www.politicshome.com/news/article/gove...   
19115  https://www.politicshome.com/news/article/coro...   

                                                   title  \
98                                                  News   
99     Zachery counting on ‘small ball’ approach from...   
100    Bobcats hope to take next step after last seas...   
101    Saylor completes full circle as 

In [6]:
df = patchedDf

# Starting error analisys

In [7]:
bertResults = pd.read_csv('Results/Classification/bert.csv')
llamaResults = pd.read_csv('Results/Classification/llama.csv')
robertaResults = pd.read_csv('Results/Classification/roberta.csv')
mistralResults = pd.read_csv('Results/Classification/mistral/24B_results.csv')



In [8]:
llamaResults 


,Fold,Train_Losses_Per_Epoch,Val_Losses_Per_Epoch,Balanced_Accuracy,Accuracy,Precision_Micro,Precision_Macro,Precision_Weighted,Recall_Micro,Recall_Macro,Recall_Weighted,F1_Micro,F1_Macro,F1_Weighted,Confusion_Matrix,Val_True_Labels,Val_Predictions
0,1,"[2.273344498424704, 0.6563329227557215, 0.3964...","[0.9109003281593323, 0.523340392112732, 0.4629...",0.546540,0.614634,0.614634,0.561393,0.617791,0.614634,0.546540,0.614634,0.614634,0.551343,0.612938,"[[776, 22, 11, 90, 18], [28, 397, 186, 142, 43...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,2,"[2.2537926341394128, 0.6427574809383138, 0.359...","[0.8524569582939148, 0.4681211179494858, 0.401...",0.542944,0.573264,0.573264,0.545999,0.610328,0.573264,0.542944,0.573264,0.573264,0.536701,0.584107,"[[729, 137, 56, 37, 28], [7, 400, 112, 49, 32]...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3, ..."
2,3,"[2.302797350053037, 0.6249480824122268, 0.3466...","[0.9556651520729065, 0.5182030880451203, 0.462...",0.521130,0.580355,0.580355,0.527020,0.605529,0.580355,0.521130,0.580355,0.580355,0.520926,0.590047,"[[840, 12, 18, 0, 11], [18, 247, 119, 78, 76],...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[4, 4, 4, 1, 3, 3, 4, 3, 1, 3, 3, 1, 3, 4, 1, ..."
3,4,"[2.351019258719099, 0.6279265853769416, 0.3830...","[0.8965884923934937, 0.5370388412475586, 0.471...",0.561900,0.591779,0.591779,0.582187,0.632612,0.591779,0.561900,0.591779,0.591779,0.567835,0.606555,"[[792, 162, 26, 95, 17], [11, 190, 12, 340, 55...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ..."
4,5,"[2.4721066037175854, 0.5970372030118464, 0.338...","[1.0018611645698547, 0.47080618143081665, 0.40...",0.569448,0.637091,0.637091,0.566040,0.637627,0.637091,0.569448,0.637091,0.637091,0.565431,0.634783,"[[946, 14, 24, 10, 0], [65, 175, 39, 104, 91],...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [9]:

robertaResults 

,Fold,Balanced_Accuracy,Accuracy,Precision_Macro,Precision_Weighted,Recall_Macro,Recall_Weighted,F1_Macro,F1_Weighted,Confusion_Matrix,Val_True_Labels,Val_Predictions
0,1,0.545132,0.607445,0.554182,0.613016,0.545132,0.607445,0.548338,0.608790,"[[770, 33, 7, 86, 21], [11, 391, 245, 115, 34]...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,2,0.533799,0.588337,0.538660,0.595666,0.533799,0.588337,0.532931,0.589032,"[[737, 80, 100, 53, 17], [11, 376, 140, 57, 16...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 1, 3, 1, 1, 3, ..."
2,3,0.533025,0.593179,0.528791,0.590510,0.533025,0.593179,0.526597,0.588478,"[[846, 14, 8, 10, 3], [38, 252, 107, 113, 28],...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[0, 3, 4, 1, 3, 2, 4, 2, 2, 3, 3, 0, 0, 4, 2, ..."
3,4,0.548994,0.601344,0.556752,0.601384,0.548994,0.601344,0.549386,0.598390,"[[899, 75, 10, 102, 6], [44, 160, 35, 356, 13]...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ..."
4,5,0.540668,0.626093,0.534656,0.605368,0.540668,0.626093,0.535598,0.613762,"[[937, 10, 27, 17, 3], [86, 142, 53, 125, 68],...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [10]:
mistralResults

,Fold,Flash_Attention,Precision_Macro,Recall_Macro,F1_Macro,Precision_micro,Recall_micro,F1_micro,Precision_weighted,Recall_weighted,F1_weighted,Accuracy,Balanced_Accuracy,MCC,Cohen_Kappa,Confusion_Matrix,Val_Article_IDs,Val_True_Labels,Val_Predictions
0,1,True,0.596051,0.561743,0.568535,0.625417,0.625417,0.625417,0.665317,0.625417,0.635112,0.625417,0.561743,0.522586,0.518896,"[[768, 9, 31, 96, 13], [15, 374, 214, 96, 97],...","['0', '1', '2', '3', '4', '5', '6', '7', '8', ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,2,True,0.551455,0.548730,0.536507,0.605387,0.605387,0.605387,0.638654,0.605387,0.610177,0.605387,0.548730,0.501691,0.495725,"[[724, 191, 38, 24, 10], [3, 455, 96, 40, 6], ...","['0', '1', '2', '3', '4', '5', '6', '7', '8', ...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 1, 1, 1, ..."
2,3,True,0.542132,0.539485,0.534869,0.600273,0.600273,0.600273,0.599504,0.600273,0.596300,0.600273,0.539485,0.487755,0.486125,"[[831, 19, 22, 4, 5], [21, 277, 102, 112, 26],...","['0', '1', '2', '3', '4', '5', '6', '7', '8', ...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[4, 4, 4, 1, 1, 3, 1, 3, 3, 3, 3, 4, 1, 4, 4, ..."
3,4,True,0.554403,0.530380,0.520401,0.591779,0.591779,0.591779,0.581802,0.591779,0.569628,0.591779,0.530380,0.478604,0.471742,"[[944, 32, 48, 63, 5], [54, 111, 92, 345, 6], ...","['0', '1', '2', '3', '4', '5', '6', '7', '8', ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ..."
4,5,True,0.564328,0.566713,0.559403,0.634091,0.634091,0.634091,0.630422,0.634091,0.624532,0.634091,0.566713,0.530551,0.526325,"[[963, 5, 17, 4, 5], [114, 151, 51, 72, 86], [...","['0', '1', '2', '3', '4', '5', '6', '7', '8', ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [11]:
import pandas as pd
import numpy as np
from ast import literal_eval
from sklearn.model_selection import StratifiedKFold

def add_predictions_and_errors(df, results_df, domain_column='domain', 
                               label_column='label', n_splits=5, random_state=42,
                               verbose=True):
    """
    Complete analysis function that reconstructs folds, adds predictions/errors,
    performs analysis, and returns all results in a structured dictionary.
    
    Parameters:
    -----------
    df: pd.DataFrame
        Original dataframe with all data
    results_df: pd.DataFrame
        Results dataframe with metrics and predictions
    domain_column: str
        Column name used for domain grouping (default: 'domain')
    label_column: str
        Column name for true labels (default: 'label')
    n_splits: int
        Number of cross-validation folds (default: 5)
    random_state: int
        Random seed for fold reconstruction (default: 42)
    verbose: bool
        Whether to print analysis output (default: True)
    
    Returns:
    --------
    dict: Dictionary containing all results with the following keys:
        - 'augmented_df': DataFrame with added columns (predicted_label, isError, fold)
        - 'error_summary': Dictionary with overall error statistics
        - 'confusion_matrix': Confusion matrix as DataFrame
        - 'error_types': DataFrame with most common error patterns
        - 'domain_analysis': DataFrame with error analysis by domain
        - 'label_analysis': DataFrame with error analysis by true label
        - 'fold_info': List of dictionaries with fold-level statistics
        - 'error_samples': DataFrame containing only misclassified samples
    """
    
    def print_header(text, width=80):
        """Helper function to print formatted headers"""
        if verbose:
            print("\n" + "="*width)
            print(text)
            print("="*width)
    
    # Initialize results dictionary
    results = {}
    
    # -----------------------------------------------------------------
    # 1. RECONSTRUCT FOLDS AND ADD PREDICTIONS
    # -----------------------------------------------------------------
    print_header("RECONSTRUCTING FOLDS AND ADDING PREDICTIONS")
    
    # Create a copy of the original dataframe
    augmented_df = df.copy()
    
    # Initialize new columns
    augmented_df['predicted_label'] = np.nan
    augmented_df['isError'] = False
    augmented_df['fold'] = np.nan
    
    # Group by domain and get most frequent label for stratification
    grouped = df.groupby(domain_column)
    domains = list(grouped.groups.keys())
    domain_labels = [group[label_column].mode()[0] for _, group in grouped]
    
    # Create StratifiedKFold with the same parameters as training
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    # Store fold information
    fold_info = []
    total_errors = 0
    total_predictions = 0
    
    if verbose:
        print(f"Reconstructing {n_splits} folds with random_state={random_state}")
        print(f"Found {len(domains)} unique domains")
    
    # Process each fold
    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(domains, domain_labels)):
        if verbose:
            print(f"\n  Processing Fold {fold_idx + 1}:")
        
        # Get test domains for this fold
        test_domains = [domains[i] for i in test_idx]
        
        # Get test samples in original dataframe
        test_mask = df[domain_column].isin(test_domains)
        test_indices = df[test_mask].index.tolist()
        
        # Get predictions from results dataframe
        true_labels_str = results_df.iloc[fold_idx]['Val_True_Labels']
        predictions_str = results_df.iloc[fold_idx]['Val_Predictions']
        
        # Convert string representation to list
        try:
            if isinstance(true_labels_str, str):
                true_labels = literal_eval(true_labels_str)
            else:
                true_labels = true_labels_str
            
            if isinstance(predictions_str, str):
                predictions = literal_eval(predictions_str)
            else:
                predictions = predictions_str
        except Exception as e:
            print(f"  Error parsing predictions for fold {fold_idx + 1}: {e}")
            continue
        
        # Verify and align lengths
        if len(true_labels) != len(test_indices):
            if verbose:
                print(f"  Warning: {len(true_labels)} predictions for {len(test_indices)} test samples")
            min_len = min(len(true_labels), len(test_indices))
            true_labels = true_labels[:min_len]
            predictions = predictions[:min_len]
            test_indices = test_indices[:min_len]
        
        # Assign predictions and errors
        fold_errors = 0
        for i, idx in enumerate(test_indices):
            true_label = true_labels[i]
            pred_label = predictions[i]
            
            augmented_df.at[idx, 'predicted_label'] = pred_label
            augmented_df.at[idx, 'isError'] = true_label != pred_label
            augmented_df.at[idx, 'fold'] = fold_idx + 1
            
            if true_label != pred_label:
                fold_errors += 1
                total_errors += 1
        
        total_predictions += len(test_indices)
        
        # Calculate fold statistics
        fold_error_rate = (fold_errors / len(test_indices)) * 100 if len(test_indices) > 0 else 0
        
        # Store fold info
        fold_info.append({
            'fold': fold_idx + 1,
            'test_domains_count': len(test_domains),
            'test_samples': len(test_indices),
            'errors': fold_errors,
            'error_rate': fold_error_rate,
            'accuracy': 100 - fold_error_rate
        })
        
        if verbose:
            print(f"    Test samples: {len(test_indices)}")
            print(f"    Errors: {fold_errors} ({fold_error_rate:.2f}%)")
            print(f"    Accuracy: {100 - fold_error_rate:.2f}%")
    
    # Convert predicted_label to integer type
    augmented_df['predicted_label'] = augmented_df['predicted_label'].astype('Int64')
    
    # Store in results
    results['augmented_df'] = augmented_df
    results['fold_info'] = fold_info
    
    # -----------------------------------------------------------------
    # 2. OVERALL ERROR SUMMARY
    # -----------------------------------------------------------------
    print_header("OVERALL ERROR SUMMARY")
    
    # Filter to rows with predictions (test set)
    test_set = augmented_df[augmented_df['predicted_label'].notna()]
    error_set = test_set[test_set['isError'] == True]
    
    overall_error_rate = (len(error_set) / len(test_set)) * 100 if len(test_set) > 0 else 0
    
    error_summary = {
        'total_samples': len(augmented_df),
        'test_samples': len(test_set),
        'train_samples': len(augmented_df) - len(test_set),
        'errors': len(error_set),
        'error_rate': overall_error_rate,
        'accuracy': 100 - overall_error_rate,
        'error_percentage': f"{overall_error_rate:.2f}%",
        'accuracy_percentage': f"{100 - overall_error_rate:.2f}%"
    }
    
    results['error_summary'] = error_summary
    
    if verbose:
        print(f"Total samples in dataset: {len(augmented_df)}")
        print(f"Samples in test set (with predictions): {len(test_set)}")
        print(f"Errors in test set: {len(error_set)}")
        print(f"Overall error rate: {overall_error_rate:.2f}%")
        print(f"Overall accuracy: {100 - overall_error_rate:.2f}%")
        
        # Compare with reported metrics
        print(f"\nReported metrics from results dataframe:")
        for i, row in results_df.iterrows():
            fold = i + 1
            reported_acc = row['Accuracy'] * 100
            reported_bal_acc = row['Balanced_Accuracy'] * 100
            print(f"  Fold {fold}: Accuracy={reported_acc:.2f}%, "
                  f"Balanced Accuracy={reported_bal_acc:.2f}%")
    
    # -----------------------------------------------------------------
    # 3. CONFUSION MATRIX
    # -----------------------------------------------------------------
    print_header("CONFUSION MATRIX (True vs Predicted)")
    
    # Create confusion matrix
    confusion_matrix = pd.crosstab(
        test_set[label_column],
        test_set['predicted_label'],
        rownames=['True'],
        colnames=['Predicted'],
        margins=True
    )
    
    results['confusion_matrix'] = confusion_matrix
    
    if verbose:
        print(confusion_matrix)
        print(f"\nTotal predictions: {confusion_matrix.loc['All', 'All']}")
    
    # -----------------------------------------------------------------
    # 4. ERROR TYPE ANALYSIS
    # -----------------------------------------------------------------
    print_header("MOST COMMON ERROR TYPES")
    
    if len(error_set) > 0:
        # Group errors by true label and predicted label
        error_types = error_set.groupby([label_column, 'predicted_label']).size().reset_index(name='count')
        error_types = error_types.sort_values('count', ascending=False)
        
        # Add percentage of total errors
        error_types['percentage'] = (error_types['count'] / len(error_set)) * 100
        
        # Add description
        error_types['error_type'] = error_types.apply(
            lambda x: f"True={x[label_column]} → Pred={x['predicted_label']}", axis=1
        )
        
        results['error_types'] = error_types
        
        if verbose:
            print(f"Total unique error patterns: {len(error_types)}")
            print(f"\nTop 10 most common error patterns:")
            for i, row in error_types.head(10).iterrows():
                print(f"  {row['error_type']}: {row['count']} samples ({row['percentage']:.1f}%)")
    else:
        results['error_types'] = pd.DataFrame()
        if verbose:
            print("No errors found in the test set!")
    
    # -----------------------------------------------------------------
    # 5. DOMAIN-LEVEL ANALYSIS
    # -----------------------------------------------------------------
    print_header("DOMAIN-LEVEL ERROR ANALYSIS")
    
    domain_stats = []
    for domain in test_set[domain_column].unique():
        domain_data = test_set[test_set[domain_column] == domain]
        if len(domain_data) > 0:
            domain_errors = domain_data['isError'].sum()
            error_rate_domain = (domain_errors / len(domain_data)) * 100
            
            # Get most common true and predicted labels for this domain
            true_mode = domain_data[label_column].mode()[0] if len(domain_data[label_column].mode()) > 0 else None
            pred_mode = domain_data['predicted_label'].mode()[0] if len(domain_data['predicted_label'].mode()) > 0 else None
            
            domain_stats.append({
                'domain': domain,
                'samples': len(domain_data),
                'errors': domain_errors,
                'error_rate': error_rate_domain,
                'accuracy': 100 - error_rate_domain,
                'most_common_true_label': true_mode,
                'most_common_pred_label': pred_mode
            })
    
    domain_analysis = pd.DataFrame(domain_stats)
    if len(domain_analysis) > 0:
        domain_analysis = domain_analysis.sort_values('error_rate', ascending=False)
        results['domain_analysis'] = domain_analysis
        
        if verbose:
            print("\nTop 10 domains with highest error rates:")
            top_domains = domain_analysis.head(10)
            for i, row in top_domains.iterrows():
                print(f"  {row['domain']}: {row['errors']}/{row['samples']} "
                      f"({row['error_rate']:.1f}% error, {row['accuracy']:.1f}% acc)")
            
            print("\nTop 10 domains with lowest error rates (minimum 5 samples):")
            low_error_domains = domain_analysis[domain_analysis['samples'] >= 5].tail(10)
            for i, row in low_error_domains.iterrows():
                print(f"  {row['domain']}: {row['errors']}/{row['samples']} "
                      f"({row['error_rate']:.1f}% error, {row['accuracy']:.1f}% acc)")
    else:
        results['domain_analysis'] = pd.DataFrame()
    
    # -----------------------------------------------------------------
    # 6. LABEL-LEVEL ANALYSIS
    # -----------------------------------------------------------------
    print_header("LABEL-LEVEL ERROR ANALYSIS")
    
    label_stats = []
    for label in sorted(test_set[label_column].unique()):
        label_data = test_set[test_set[label_column] == label]
        label_errors = label_data['isError'].sum()
        error_rate_label = (label_errors / len(label_data)) * 100 if len(label_data) > 0 else 0
        
        # Most common mispredictions for this label
        label_errors_data = label_data[label_data['isError'] == True]
        if len(label_errors_data) > 0:
            mispred_counts = label_errors_data['predicted_label'].value_counts()
            mispred_str = ", ".join([f"{pred}({count})" for pred, count in mispred_counts.head(3).items()])
        else:
            mispred_str = "None"
        
        # Distribution of predictions for this true label
        pred_dist = label_data['predicted_label'].value_counts(normalize=True).head(3)
        pred_dist_str = ", ".join([f"{pred}({pct:.1%})" for pred, pct in pred_dist.items()])
        
        label_stats.append({
            'true_label': label,
            'samples': len(label_data),
            'errors': label_errors,
            'error_rate': error_rate_label,
            'accuracy': 100 - error_rate_label,
            'most_common_mispredictions': mispred_str,
            'prediction_distribution': pred_dist_str
        })
    
    label_analysis = pd.DataFrame(label_stats)
    label_analysis = label_analysis.sort_values('error_rate', ascending=False)
    results['label_analysis'] = label_analysis
    
    if verbose:
        print("\nError rates by true label:")
        for i, row in label_analysis.iterrows():
            print(f"  Label {row['true_label']}: {row['errors']}/{row['samples']} "
                  f"({row['error_rate']:.1f}% error, {row['accuracy']:.1f}% acc)")
            if row['most_common_mispredictions'] != "None":
                print(f"    Common mispredictions: {row['most_common_mispredictions']}")
            print(f"    Prediction distribution: {row['prediction_distribution']}")
    
    # -----------------------------------------------------------------
    # 7. ERROR SAMPLES DATAFRAME
    # -----------------------------------------------------------------
    if len(error_set) > 0:
        error_samples = error_set.copy()
        # Add useful columns for analysis
        error_samples['true_label'] = error_samples[label_column]
        error_samples['pred_label'] = error_samples['predicted_label']
        
        # Keep only relevant columns
        keep_cols = ['url', 'title', 'text', 'authors', domain_column, 
                     'true_label', 'pred_label', 'fold', 'isError']
        # Only keep columns that exist
        available_cols = [col for col in keep_cols if col in error_samples.columns]
        error_samples = error_samples[available_cols]
        
        results['error_samples'] = error_samples
    else:
        results['error_samples'] = pd.DataFrame()
    
    # -----------------------------------------------------------------
    # 8. FOLD-LEVEL SUMMARY
    # -----------------------------------------------------------------
    print_header("FOLD-LEVEL SUMMARY")
    
    fold_summary_df = pd.DataFrame(fold_info)
    results['fold_summary'] = fold_summary_df
    
    if verbose:
        print("\nFold statistics:")
        print(fold_summary_df.to_string(index=False))
        
        # Calculate average across folds
        avg_error_rate = fold_summary_df['error_rate'].mean()
        avg_accuracy = fold_summary_df['accuracy'].mean()
        print(f"\nAverage across folds:")
        print(f"  Error rate: {avg_error_rate:.2f}%")
        print(f"  Accuracy: {avg_accuracy:.2f}%")
    
    # -----------------------------------------------------------------
    # 9. FINAL SUMMARY
    # -----------------------------------------------------------------
    print_header("ANALYSIS COMPLETE")
    
    if verbose:
        print(f"\nResults available in returned dictionary with keys:")
        for key in results.keys():
            print(f"  - '{key}'")
        
        print(f"\nTotal errors identified: {len(error_set)}")
        print(f"Overall test accuracy: {100 - overall_error_rate:.2f}%")
    
    return results

# Example usage function
def example_usage():
    """
    Example of how to use the function
    """
    # Assuming you have df and results_df loaded
    # bertErrors = add_predictions_and_errors(df, robertaResults)
    
    # Access different parts of the results:
    # 1. Get augmented dataframe
    # augmented_df = bertErrors['augmented_df']
    
    # 2. Get error summary
    # summary = bertErrors['error_summary']
    # print(f"Overall accuracy: {summary['accuracy']:.2f}%")
    
    # 3. Get confusion matrix
    # confusion = bertErrors['confusion_matrix']
    
    # 4. Get error samples for detailed inspection
    # error_samples = bertErrors['error_samples']
    # if len(error_samples) > 0:
    #     print(f"Found {len(error_samples)} errors")
    #     print(error_samples.head())
    
    pass

# Quick utility function to save all results
def save_analysis_results(errorsDf, output_prefix='bert_analysis'):
    """
    Save all analysis results to CSV files
    """
    # Save augmented dataframe
    bertErrors['augmented_df'].to_csv(f'{output_prefix}_augmented.csv', index=False)
    
    # Save error samples
    if len(errorsDf['error_samples']) > 0:
        errorsDf['error_samples'].to_csv(f'{output_prefix}_errors.csv', index=False)
    
    # Save confusion matrix
    errorsDf['confusion_matrix'].to_csv(f'{output_prefix}_confusion.csv')
    
    # Save error types
    if len(errorsDf['error_types']) > 0:
        errorsDf['error_types'].to_csv(f'{output_prefix}_error_types.csv', index=False)
    
    # Save domain analysis
    if len(errorsDf['domain_analysis']) > 0:
        errorsDf['domain_analysis'].to_csv(f'{output_prefix}_domain_analysis.csv', index=False)
    
    # Save label analysis
    errorsDf['label_analysis'].to_csv(f'{output_prefix}_label_analysis.csv', index=False)
    
    # Save fold summary
    errorsDf['fold_summary'].to_csv(f'{output_prefix}_fold_summary.csv', index=False)
    
    print(f"All results saved with prefix: {output_prefix}_")

In [12]:
# Or with custom parameters
robertaErrors = add_predictions_and_errors(
    df=df,
    results_df=robertaResults,
    domain_column='domain',
    label_column='label',
    n_splits=5,
    random_state=42,
    verbose=False
)
robertaErrors = robertaErrors['augmented_df']

In [13]:
# Or with custom parameters
llamaErrors = add_predictions_and_errors(
    df=df,
    results_df=llamaResults,
    domain_column='domain',
    label_column='label',
    n_splits=5,
    random_state=42,
    verbose=False
)
llamaErrors = llamaErrors['augmented_df']

In [14]:
bertResults= bertResults.rename(columns={'True_Labels': 'Val_True_Labels', 'Predicted_Labels': 'Val_Predictions'})


# Or with custom parameters
bertErrors = add_predictions_and_errors(
    df=df,
    results_df=bertResults,
    domain_column='domain',
    label_column='label',
    n_splits=5,
    random_state=42,
    verbose=False
)
bertErrors = bertErrors['augmented_df']

In [15]:
# Or with custom parameters
mistralErrors = add_predictions_and_errors(
    df=df,
    results_df=mistralResults,
    domain_column='domain',
    label_column='label',
    n_splits=5,
    random_state=42,
    verbose=False
)
mistralErrors = mistralErrors['augmented_df']

## Old paper plots

In [16]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from typing import List, Tuple
from matplotlib.patches import Rectangle


def plot_errors_combined_chart(
    errors_dfs: List[pd.DataFrame],
    model_names: List[str],
    orientation_col: str = "Orientation",
    label_col: str = "label",
    predicted_col: str = "predicted_label",
    figsize: tuple = (12, 8),
    show: bool = True,
    save_path: str = None,
    print_df: bool = True,
) -> Tuple[plt.Figure, pd.DataFrame]:
    """
    Plots error counts for different models across distances and orientations.
    Distance is calculated as |true_label - predicted_label| for reputation scores (1-5).

    Args:
        errors_dfs (List[pd.DataFrame]): List of DataFrames with error data for each model.
                                         Each DataFrame must have 'Orientation', 'label', 
                                         'predicted_label', and 'isError' columns.
        model_names (List[str]): List of names corresponding to the models in errors_dfs.
        orientation_col (str): The name of the column representing orientation.
        label_col (str): The name of the column with true labels (reputation 1-5).
        predicted_col (str): The name of the column with predicted labels.
        figsize (tuple): Figure size (width, height) in inches.
        show (bool): If True, displays the plot.
        save_path (str): If provided, saves the plot to this path.
        print_df (bool): If True, prints the aggregated DataFrame to the console output.

    Returns:
        Tuple[plt.Figure, pd.DataFrame]: A tuple containing the Matplotlib Figure object
                                         and the Pandas DataFrame with the aggregated error counts.
    """
    if len(errors_dfs) != len(model_names):
        raise ValueError("Length of errors_dfs and model_names must match.")

    # Get all unique orientations
    all_orientations = sorted(set().union(*[df[orientation_col].unique() for df in errors_dfs]))

    base_colors = {
        "BERT": "tab:blue",
        "Llama": "tab:orange",
        "Roberta": "tab:green",
        "MentalBERT": "tab:red",
        "GPT": "tab:purple",
    }
    hatch_patterns = ['///', '\\\\\\', 'xxx', '---', '+++', '...', 'ooo', '***', '|||', '///\\\\\\']

    # Aggregate error counts with calculated distance
    bar_data = []
    for model_name, df in zip(model_names, errors_dfs):
        # Filter only errors (isError == True)
        error_df = df[df['isError'] == True].copy()
        
        # Calculate distance as absolute difference between true and predicted label
        error_df['distance'] = (error_df[label_col] - error_df[predicted_col]).abs()
        
        # Group by distance and orientation
        grouped = (
            error_df.groupby(['distance', orientation_col])
            .size()
            .reset_index(name="n_errors")
        )
        
        # Get all distances found in this model's errors
        model_distances = sorted(error_df['distance'].unique())
        
        for dist in model_distances:
            for ori in all_orientations:
                count = grouped.query(f"distance == @dist and {orientation_col} == @ori")["n_errors"]
                n_errors = int(count.iloc[0]) if not count.empty else 0
                if n_errors > 0:  # Only add if there are errors
                    bar_data.append({
                        "Model": model_name,
                        "Distance": dist,
                        "Orientation": ori,
                        "Errors": n_errors
                    })

    plot_df = pd.DataFrame(bar_data)
    
    # Get all distances across all models
    all_distances = sorted(plot_df['Distance'].unique())

    if print_df:
        print("\nAggregated Error Data for Plot:")
        print("Distance = |true_label - predicted_label|")
        pivot_df = plot_df.pivot_table(
            index=['Distance', 'Orientation'],
            columns='Model',
            values='Errors',
            fill_value=0
        )
        print(pivot_df)
        print("-" * 50)

    total_bars_per_dist = len(model_names) * len(all_orientations)
    bar_width = 0.8 / total_bars_per_dist

    fig, ax = plt.subplots(figsize=figsize)

    # Plot bars
    for m_idx, model in enumerate(model_names):
        base_color = base_colors.get(model, f"C{m_idx}")
        for o_idx, ori in enumerate(all_orientations):
            hatch = hatch_patterns[o_idx % len(hatch_patterns)]

            x_vals, y_vals = [], []
            for dist in all_distances:
                subset = plot_df.query("Model == @model and Orientation == @ori and Distance == @dist")
                errors = subset["Errors"].values[0] if not subset.empty else 0
                offset = (m_idx * len(all_orientations) + o_idx)
                x_pos = dist + (offset - total_bars_per_dist / 2) * bar_width + bar_width / 2
                x_vals.append(x_pos)
                y_vals.append(errors)

            bars = ax.bar(
                x_vals, y_vals, width=bar_width,
                color=base_color, hatch=hatch, edgecolor='black'
            )

    # Remove default x-axis labels
    ax.set_xlabel("")
    ax.set_xticks([])
    
    # Add distance labels at the top
    y_max = ax.get_ylim()[1]
    for dist in all_distances:
        max_error_for_dist = plot_df[plot_df['Distance'] == dist]['Errors'].max()
        ax.text(dist, max_error_for_dist + 0.02 * y_max, f"Distance: {int(dist)}",
                ha='center', va='bottom', fontsize=12, fontweight='bold')

    # Create x-tick labels for models
    new_xticks = []
    new_xticklabels = []
    model_x_ranges = {model: [] for model in model_names}
    x_label_offset = bar_width / 2

    for dist_idx, dist in enumerate(all_distances):
        for m_idx, model in enumerate(model_names):
            num_orientations = len(all_orientations)
            offset_first = m_idx * num_orientations
            offset_last = (m_idx + 1) * num_orientations - 1
            center_offset = (offset_first + offset_last) / 2.0
            x_pos = dist + (center_offset - total_bars_per_dist / 2) * bar_width + bar_width / 2
            new_xticks.append(x_pos + x_label_offset)
            new_xticklabels.append(model)
            
            x_start_model_group = dist + (offset_first - total_bars_per_dist / 2) * bar_width
            x_end_model_group = dist + (offset_last + 1 - total_bars_per_dist / 2) * bar_width
            model_x_ranges[model].append((x_start_model_group, x_end_model_group))
            
            # Add separator lines between models
            if m_idx < len(model_names) - 1:
                separator_x = dist + ((offset_last + 1) - total_bars_per_dist / 2) * bar_width
                ax.axvline(separator_x, color='gray', linestyle='--', linewidth=0.8, zorder=0)

    ax.set_xticks(new_xticks)
    ax.set_xticklabels(new_xticklabels, rotation=45, ha="center", fontsize=13, fontweight='bold')
    ax.tick_params(axis='x', which='major', pad=5)

    # Add alternating background for model groups
    for dist_idx, dist in enumerate(all_distances):
        for m_idx, model in enumerate(model_names):
            x_start, x_end = model_x_ranges[model][dist_idx]
            if m_idx % 2 == 0:
                ax.axvspan(x_start, x_end, facecolor='lightgray', alpha=0.2, zorder=-1)
            else:
                ax.axvspan(x_start, x_end, facecolor='white', alpha=0.2, zorder=-1)

    # Create custom legend with hatches
    custom_legend_handles = []
    for o_idx, ori in enumerate(all_orientations):
        hatch = hatch_patterns[o_idx % len(hatch_patterns)]
        custom_legend_handles.append(
            Rectangle((0, 0), 0.5, 2, fc="white", hatch=hatch, edgecolor='black', label=ori)
        )

    legend = ax.legend(
        handles=custom_legend_handles,
        title="Leaning",
        loc='upper right',
        bbox_to_anchor=(1, 1),
        frameon=True,
        fontsize=15,
        labelspacing=1.2,
        handlelength=1,
    )
    plt.setp(legend.get_title(), fontsize=19)

    ax.set_ylabel("Number of Misclassifications", fontsize=13)
    ax.tick_params(axis='y', labelsize=12)

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.15)

    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"📁 Plot saved to: {save_path}")

    if show:
        plt.show()
    plt.savefig('Results/Plots/Error-Chart')

    return fig, plot_df


In [17]:
fig = plot_errors_combined_chart(
    errors_dfs=[bertErrors, llamaErrors, robertaErrors],
    model_names=["BERT", "Llama", "Roberta"],
    save_path="Results/Errors/Error-Chart"
)


Aggregated Error Data for Plot:
Distance = |true_label - predicted_label|
Model                      BERT   Llama  Roberta
Distance Orientation                            
1        Far Left         266.0   293.0    302.0
         Far Right        771.0   755.0    776.0
         Non labelled    1229.0  1195.0   1101.0
         Slightly Left    899.0   716.0    734.0
         Slightly Right  1469.0  1677.0   1506.0
2        Far Left         268.0   178.0    194.0
         Far Right        414.0   396.0    375.0
         Non labelled     630.0   640.0    691.0
         Slightly Left    375.0   291.0    360.0
         Slightly Right   671.0   531.0    605.0
3        Far Left         172.0   254.0    227.0
         Far Right        228.0   132.0    175.0
         Non labelled     218.0   226.0    224.0
         Slightly Left     96.0   128.0    136.0
         Slightly Right   251.0   284.0    212.0
4        Far Left          10.0     5.0      4.0
         Far Right         51.0    50.0    

/tmp/ipykernel_249192/957568323.py:208: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from typing import List, Tuple
from matplotlib.patches import Patch, Rectangle

def plot_orientation_distribution_per_label(
    df: pd.DataFrame,
    label_col: str = 'label',
    orientation_col: str = 'Orientation',
    figsize: tuple = (10, 6),
    show: bool = True,
    save_path: str = None,
    print_df: bool = True
) -> Tuple[plt.Figure, pd.DataFrame]:
    """
    Plots the distribution of orientation values across different labels.
    Colors represent different orientations, not reliability labels.
    
    - Asse X rinominato in "Reliability label".
    - Asse Y rinominato in "N° of articles".
    - Label asse X partono da 1 e includono le fasce di percentuale.
    - Ordine Orientation personalizzato.
    - Colori assegnati per Orientation, non per Label.

    Args:
        df (pd.DataFrame): The input DataFrame containing 'label' and 'Orientation' columns.
        label_col (str): The name of the column containing the labels (e.g., 'label').
        orientation_col (str): The name of the column containing the orientation values.
        figsize (tuple): Figure size (width, height) in inches.
        show (bool): If True, displays the plot.
        save_path (str): If provided, saves the plot to this path.
        print_df (bool): If True, prints the aggregated DataFrame to the console output.

    Returns:
        Tuple[plt.Figure, pd.DataFrame]: A tuple containing the Matplotlib Figure object
                                          and the Pandas DataFrame with the aggregated data.
    """

    if label_col not in df.columns:
        raise ValueError(f"'{label_col}' column not found in the DataFrame.")
    if orientation_col not in df.columns:
        raise ValueError(f"'{orientation_col}' column not found in the DataFrame.")

    all_labels = sorted(df[label_col].unique())
    
    # Mappa per le fasce di reliability
    reliability_ranges = {
        0: "1 (0-39)",
        1: "2 (40-59)",
        2: "3 (60-74)",
        3: "4 (75-99)",
        4: "5 (100)"
    }
    
    # --- Ordine personalizzato delle Orientation ---
    desired_order = [
        "Far Left", "Slightly Left", "Non labelled", "Slightly Right", "Far Right"
    ]
    
    # Ottieni le orientation presenti nel dataframe
    present_orientations = set(df[orientation_col].unique())
    
    # Sostituisci "Non labelled" con "Non labelled" nel dataframe
    df[orientation_col] = df[orientation_col].replace("Non labelled", "Non labelled")
    present_orientations = set(df[orientation_col].unique())
    
    # Costruisci la lista ordinata: 
    # 1. Prima quelli nell'ordine desiderato (se esistono nei dati)
    all_orientations = [x for x in desired_order if x in present_orientations]
    
    # 2. Aggiungi eventuali valori presenti nei dati ma non nella lista desiderata (per sicurezza)
    others = [x for x in present_orientations if x not in desired_order]
    all_orientations.extend(sorted(others))
    # ---------------------------------------------------------

    # Define distinct colors for each orientation
    orientation_colors = [
        'tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple',
        'tab:brown', 'tab:pink', 'tab:gray', 'tab:olive', 'tab:cyan'
    ]
    color_map = {ori: orientation_colors[i % len(orientation_colors)] for i, ori in enumerate(all_orientations)}
    
    # Define distinct hatch patterns for accessibility (colorblind, b&w printing)
    hatch_patterns = ['///', '\\\\\\', 'xxx', '---', '+++', '...', 'ooo', '***', '|||', '///\\\\\\']
    
    if len(all_orientations) > len(hatch_patterns):
        print("Warning: Not enough unique hatch patterns for all orientations. Patterns will repeat.")
        hatch_patterns = (hatch_patterns * ((len(all_orientations) // len(hatch_patterns)) + 1))[:len(all_orientations)]
    
    hatch_map = {ori: hatch_patterns[i % len(hatch_patterns)] for i, ori in enumerate(all_orientations)}

    # Prepare data for plotting and for the output DataFrame
    raw_plot_data = [] 
    for label in all_labels:
        subset = df[df[label_col] == label]
        # Reindex using the specific sorted order to ensure correct counting structure
        orientation_counts = subset[orientation_col].value_counts().reindex(all_orientations, fill_value=0)
        
        for ori in all_orientations:
            count = orientation_counts.get(ori, 0)
            raw_plot_data.append({
                'Label': label,
                'Orientation': ori,
                'Count': count
            })
            
    plot_df = pd.DataFrame(raw_plot_data)

    # Optionally print the DataFrame
    if print_df:
        print("\nAggregated Data for Plot:")
        plot_df['Orientation'] = pd.Categorical(plot_df['Orientation'], categories=all_orientations, ordered=True)
        pivot_df = plot_df.pivot_table(index='Orientation', columns='Label', values='Count', fill_value=0)
        print(pivot_df)
        print("-" * 50)

    fig, ax = plt.subplots(figsize=figsize)

    num_orientations = len(all_orientations)
    group_total_width = 0.8 
    bar_width = group_total_width / num_orientations
    
    x_positions = np.arange(len(all_labels))

    # Plot bars with colors based on orientation
    for l_idx, label in enumerate(all_labels):
        subset_data = plot_df.query("Label == @label and Count > 0")
        
        # Filtra mantenendo l'ordine corretto di 'all_orientations'
        present_oris_with_indices = [
            ori for ori in all_orientations if ori in subset_data['Orientation'].values
        ]

        num_present_orientations = len(present_oris_with_indices)

        if num_present_orientations == 0:
            continue

        for k, ori in enumerate(present_oris_with_indices):
            # Calcola il nuovo offset per centrare le barre *presenti*
            offset = (k - num_present_orientations / 2 + 0.5) * bar_width
            
            # Usa il colore e l'hatch basati sull'orientation
            current_color = color_map[ori]
            current_hatch = hatch_map[ori]
            
            height = subset_data[subset_data["Orientation"] == ori]['Count'].values[0]
            
            ax.bar(
                x_positions[l_idx] + offset,
                height,
                width=bar_width,
                color=current_color,
                edgecolor='black',
                linewidth=1.2,
                hatch=current_hatch
            )
            
    # --- Rinomina Assi ---
    ax.set_ylabel("N° of articles", fontsize=14)
    ax.set_xlabel("Reliability label", fontsize=14)
    
    ax.set_xticks(x_positions)
    # --- Etichette X partono da 1 e includono le fasce di percentuale ---
    x_labels = [reliability_ranges.get(i + 1, str(i + 1)) for i in x_positions]
    ax.set_xticklabels(x_labels, rotation=0, ha='center', fontsize=12, fontweight='bold')
    
    ax.tick_params(axis='y', labelsize=12)

    # Creazione della legenda per le Orientations (colori + hatch patterns)
    orientation_legend_handles = []
    for ori in all_orientations:
        orientation_legend_handles.append(
            Patch(facecolor=color_map[ori], edgecolor='black', linewidth=1.2, hatch=hatch_map[ori], label=str(ori))
        )
        
    orientation_legend = ax.legend(
        handles=orientation_legend_handles,
        title='Leaning',
        loc='upper right',
        frameon=True,
        edgecolor='black',
        fontsize=12,
        title_fontsize=14,
        handlelength=2.0,
        handleheight=2.0
    )

    ax.add_artist(orientation_legend)

    plt.tight_layout()

    if save_path:
        try:
            fig.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"📁 Plot saved to: {save_path}")
        except Exception as e:
            print(f"Error saving plot: {e}")

    if show:
        plt.show()
    
    return fig, plot_df

# Esempio di utilizzo
fig, data = plot_orientation_distribution_per_label(
    df,
    save_path="Results/Plots/orientationDistribution" 
)


Aggregated Data for Plot:
Label                0      1       2       3      4
Orientation                                         
Far Left          86.0  355.0     6.0   408.0  160.0
Slightly Left      0.0  245.0   174.0  1325.0  663.0
Non labelled      71.0  635.0  3189.0  1964.0  563.0
Slightly Right  4245.0  782.0   960.0   961.0  542.0
Far Right        469.0  999.0   291.0   383.0    0.0
--------------------------------------------------


/tmp/ipykernel_249192/2607006898.py:114: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_df = plot_df.pivot_table(index='Orientation', columns='Label', values='Count', fill_value=0)


📁 Plot saved to: Results/Plots/orientationDistribution


/tmp/ipykernel_249192/2607006898.py:202: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()


In [19]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import math
from matplotlib.patches import Rectangle
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

def plot_confusion_matrix_detailed_square_domains(
    df, print_text=True, modelName='NoModelName'
):
    plt.rcParams.update({
        "text.usetex": False,
        "font.family": "serif",
        "mathtext.fontset": "cm",
    })

    labelMapping = {0:r"$\mathit{l1}$",1:r"$\mathit{l2}$",2:r"$\mathit{l3}$",3:r"$\mathit{l4}$",4:r"$\mathit{l5}$"}

    df = df.copy()
    df['Orientation'] = df['Orientation'].replace("Non labelled", "Non labelled")
    all_labels = sorted(df['label'].unique())
    orientations = sorted(df['Orientation'].unique())
    n_orientations = len(orientations)
    n_classes = len(all_labels)

    # vmax globale
    max_error_pct_global = 0
    for orientation in orientations:
        temp_data = df[df['Orientation'] == orientation]
        temp_cm = pd.crosstab(temp_data['label'], temp_data['predicted_label']).reindex(index=all_labels, columns=all_labels, fill_value=0)
        temp_row_sums = temp_cm.sum(axis=1)
        for i in range(n_classes):
            for j in range(n_classes):
                if i!=j and temp_row_sums.iloc[i] > 0:
                    pct = temp_cm.iloc[i, j] / temp_row_sums.iloc[i] * 100
                    max_error_pct_global = max(max_error_pct_global, pct)
    vmax_global = max(max_error_pct_global, 10)

    # plot
    n_cols = 3
    n_rows = math.ceil(n_orientations / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(24, 9 * n_rows))
    axes_flat = axes.flatten() if n_orientations > 1 else [axes]

    textual_rows = []

    for idx, orientation in enumerate(orientations):
        ax = axes_flat[idx]
        orient_data = df[df['Orientation'] == orientation]
        total_samples = len(orient_data)
        total_errors = len(orient_data[orient_data['isError'] == 1])
        accuracy = (1 - total_errors/total_samples)*100 if total_samples>0 else 0

        cm = pd.crosstab(orient_data['label'], orient_data['predicted_label']).reindex(index=all_labels, columns=all_labels, fill_value=0)
        row_sums = cm.sum(axis=1)

        color_matrix = pd.DataFrame(index=all_labels, columns=all_labels, dtype=float)
        annot_matrix = pd.DataFrame(index=all_labels, columns=all_labels, dtype=object)
        empty_rows = []

        for i, true_label in enumerate(all_labels):
            total_class = row_sums[true_label]
            row_subset = orient_data[orient_data['label'] == true_label]
            total_domains_in_row = row_subset['domain'].nunique()

            if total_class == 0:
                empty_rows.append(i)
                continue

            for j, pred_label in enumerate(all_labels):
                count = cm.iloc[i, j]
                pct_of_class = (count / total_class * 100) if total_class>0 else 0
                cell_subset = orient_data[(orient_data['label']==true_label) & (orient_data['predicted_label']==pred_label)]
                n_dom = cell_subset['domain'].nunique()

                # salva per output testuale
                textual_rows.append({
                    "Orientation": orientation,
                    "True": labelMapping[true_label],
                    "Pred": labelMapping[pred_label],
                    "Count": int(count),
                    "Pct_True": round(pct_of_class,1),
                    "Pct_Tot": round(count/total_samples*100,1),
                    "Domains": f"{n_dom}/{total_domains_in_row}",
                    "Correct": "✓" if true_label==pred_label else "✗"
                })

                cell_text = f"{pct_of_class:.1f}%\n{count}/{total_class}\n{count/total_samples:.1%}\nDom: {n_dom}/{total_domains_in_row}"
                if i==j:
                    color_matrix.iloc[i,j]=np.nan; annot_matrix.iloc[i,j]=cell_text
                else:
                    color_matrix.iloc[i,j]=pct_of_class if count>0 else np.nan
                    annot_matrix.iloc[i,j]=cell_text if count>0 else f"0\nDom: 0/{total_domains_in_row}"

        # heatmap (codice originale invariato)
        mask = np.isnan(color_matrix.values)
        sns.heatmap(color_matrix, cmap='plasma', mask=mask, vmin=0, vmax=vmax_global, linewidths=1, linecolor='gray', square=True, ax=ax, cbar=False)
        for i in range(n_classes):
            for j in range(n_classes):
                if i in empty_rows: continue
                txt = annot_matrix.iloc[i,j]
                bg = color_matrix.iloc[i,j]
                color = 'white' if (not np.isnan(bg) and bg < vmax_global*0.5) else 'black'
                if i==j: color='black'
                ax.text(j+0.5, i+0.5, txt, ha='center', va='center', color=color, fontsize=11, fontweight='bold' if i==j or (not np.isnan(bg)) else 'normal')

        ax.set_title(f"Orientation: {orientation}\n(Diag: Accuracy | Off-Diag: % Error)", fontsize=20, fontweight='bold', pad=15)
        ax.set_xlabel("Predicted Class", fontsize=16, fontweight='bold'); ax.set_ylabel("True Class", fontsize=16, fontweight='bold')
        ax.set_xticks(np.arange(n_classes)+0.5); ax.set_yticks(np.arange(n_classes)+0.5)
        ax.set_xticklabels([labelMapping[i] for i in all_labels], fontsize=14)
        ax.set_yticklabels([labelMapping[i] for i in all_labels], fontsize=14)

    for i in range(n_orientations, len(axes_flat)): axes_flat[i].axis('off')
    plt.tight_layout(); plt.subplots_adjust(top=0.94, bottom=0.1, hspace=0.4, wspace=0.2, right=0.88)
    cbar_ax = fig.add_axes([0.90, 0.15, 0.015, 0.7])
    sm = ScalarMappable(norm=Normalize(vmin=0, vmax=vmax_global), cmap='plasma'); sm.set_array([])
    fig.colorbar(sm, cax=cbar_ax).set_label('Global % Error Rate', fontsize=16, fontweight='bold')
    plt.savefig(f'Results/Plots/cmOnPoliticalLeaning{modelName}.pdf', bbox_inches='tight')
    plt.show()

    # ------------------- OUTPUT TESTUALE -------------------
    if print_text:
        summary_df = pd.DataFrame(textual_rows)

        print("\n" + "="*90)
        print(f"OUTPUT TESTUALE - Matrice dettagliata {modelName}")
        print("="*90)

        for orientation in orientations:
            sub = summary_df[summary_df["Orientation"]==orientation]
            total = sub["Count"].sum()
            correct = sub[sub["Correct"]=="✓"]["Count"].sum()
            acc = correct/total*100 if total>0 else 0

            print(f"\n>>> ORIENTATION: {orientation} | Samples: {total} | Accuracy: {acc:.1f}%")
            print("-"*90)
            # stampa solo celle non vuote
            to_print = sub[sub["Count"]>0][["True","Pred","Count","Pct_True","Pct_Tot","Domains","Correct"]]
            print(to_print.to_string(index=False))

    return fig

In [20]:
# --- ESEMPIO ---
if 'llamaErrors' in locals():
    import os
    os.makedirs('Results/Plots', exist_ok=True)
    fig = plot_confusion_matrix_detailed_square_domains(llamaErrors, print_text=True, modelName='LLama')
    plt.show()


OUTPUT TESTUALE - Matrice dettagliata LLama

>>> ORIENTATION: Far Left | Samples: 1015 | Accuracy: 28.1%
------------------------------------------------------------------------------------------
         True          Pred  Count  Pct_True  Pct_Tot Domains Correct
$\mathit{l1}$ $\mathit{l1}$      2       2.3      0.2     1/1       ✓
$\mathit{l1}$ $\mathit{l2}$      3       3.5      0.3     1/1       ✗
$\mathit{l1}$ $\mathit{l3}$      2       2.3      0.2     1/1       ✗
$\mathit{l1}$ $\mathit{l4}$     79      91.9      7.8     1/1       ✗
$\mathit{l2}$ $\mathit{l1}$     19       5.4      1.9     4/6       ✗
$\mathit{l2}$ $\mathit{l2}$     98      27.6      9.7     6/6       ✓
$\mathit{l2}$ $\mathit{l3}$     27       7.6      2.7     3/6       ✗
$\mathit{l2}$ $\mathit{l4}$    100      28.2      9.9     5/6       ✗
$\mathit{l2}$ $\mathit{l5}$    111      31.3     10.9     5/6       ✗
$\mathit{l3}$ $\mathit{l2}$      3      50.0      0.3     1/1       ✗
$\mathit{l3}$ $\mathit{l4}$      

/tmp/ipykernel_249192/1456492331.py:121: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_249192/747241549.py:6: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()


In [21]:
# --- USAGE EXAMPLE ---
if 'bertErrors' in locals():
    fig = plot_confusion_matrix_detailed_square_domains(bertErrors, print_text=True, modelName='Bert')
    plt.show()


OUTPUT TESTUALE - Matrice dettagliata Bert

>>> ORIENTATION: Far Left | Samples: 1015 | Accuracy: 29.5%
------------------------------------------------------------------------------------------
         True          Pred  Count  Pct_True  Pct_Tot Domains Correct
$\mathit{l1}$ $\mathit{l2}$     15      17.4      1.5     1/1       ✗
$\mathit{l1}$ $\mathit{l3}$     10      11.6      1.0     1/1       ✗
$\mathit{l1}$ $\mathit{l4}$     54      62.8      5.3     1/1       ✗
$\mathit{l1}$ $\mathit{l5}$      7       8.1      0.7     1/1       ✗
$\mathit{l2}$ $\mathit{l1}$     22       6.2      2.2     2/6       ✗
$\mathit{l2}$ $\mathit{l2}$     77      21.7      7.6     6/6       ✓
$\mathit{l2}$ $\mathit{l3}$     36      10.1      3.5     4/6       ✗
$\mathit{l2}$ $\mathit{l4}$    135      38.0     13.3     6/6       ✗
$\mathit{l2}$ $\mathit{l5}$     85      23.9      8.4     4/6       ✗
$\mathit{l3}$ $\mathit{l2}$      1      16.7      0.1     1/1       ✗
$\mathit{l3}$ $\mathit{l4}$      4

/tmp/ipykernel_249192/1456492331.py:121: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_249192/1661829796.py:4: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()


In [22]:
# --- USAGE EXAMPLE ---
if 'robertaErrors' in locals():
    fig = plot_confusion_matrix_detailed_square_domains(robertaErrors, print_text=True, modelName='Roberta')
    plt.show()


OUTPUT TESTUALE - Matrice dettagliata Roberta

>>> ORIENTATION: Far Left | Samples: 1015 | Accuracy: 28.4%
------------------------------------------------------------------------------------------
         True          Pred  Count  Pct_True  Pct_Tot Domains Correct
$\mathit{l1}$ $\mathit{l1}$      3       3.5      0.3     1/1       ✓
$\mathit{l1}$ $\mathit{l2}$      5       5.8      0.5     1/1       ✗
$\mathit{l1}$ $\mathit{l4}$     78      90.7      7.7     1/1       ✗
$\mathit{l2}$ $\mathit{l1}$     24       6.8      2.4     4/6       ✗
$\mathit{l2}$ $\mathit{l2}$     72      20.3      7.1     5/6       ✓
$\mathit{l2}$ $\mathit{l3}$     51      14.4      5.0     5/6       ✗
$\mathit{l2}$ $\mathit{l4}$    129      36.3     12.7     5/6       ✗
$\mathit{l2}$ $\mathit{l5}$     79      22.3      7.8     5/6       ✗
$\mathit{l3}$ $\mathit{l2}$      2      33.3      0.2     1/1       ✗
$\mathit{l3}$ $\mathit{l4}$      4      66.7      0.4     1/1       ✗
$\mathit{l4}$ $\mathit{l1}$    

/tmp/ipykernel_249192/1456492331.py:121: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_249192/3646643048.py:4: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()


In [23]:
# --- USAGE EXAMPLE ---
if 'mistralErrors' in locals():
    fig = plot_confusion_matrix_detailed_square_domains(mistralErrors, print_text=True, modelName='Mistral')
    plt.show()


OUTPUT TESTUALE - Matrice dettagliata Mistral

>>> ORIENTATION: Far Left | Samples: 1015 | Accuracy: 36.5%
------------------------------------------------------------------------------------------
         True          Pred  Count  Pct_True  Pct_Tot Domains Correct
$\mathit{l1}$ $\mathit{l3}$      2       2.3      0.2     1/1       ✗
$\mathit{l1}$ $\mathit{l4}$     83      96.5      8.2     1/1       ✗
$\mathit{l1}$ $\mathit{l5}$      1       1.2      0.1     1/1       ✗
$\mathit{l2}$ $\mathit{l1}$     45      12.7      4.4     3/6       ✗
$\mathit{l2}$ $\mathit{l2}$     92      25.9      9.1     6/6       ✓
$\mathit{l2}$ $\mathit{l3}$     36      10.1      3.5     3/6       ✗
$\mathit{l2}$ $\mathit{l4}$     79      22.3      7.8     6/6       ✗
$\mathit{l2}$ $\mathit{l5}$    103      29.0     10.1     5/6       ✗
$\mathit{l3}$ $\mathit{l3}$      5      83.3      0.5     1/1       ✓
$\mathit{l3}$ $\mathit{l4}$      1      16.7      0.1     1/1       ✗
$\mathit{l4}$ $\mathit{l1}$    

/tmp/ipykernel_249192/1456492331.py:121: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_249192/2678580534.py:4: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()


In [24]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from typing import List, Tuple
from matplotlib.patches import Patch, Rectangle

def plot_orientation_distribution_per_label(
    df: pd.DataFrame,
    label_col: str = 'label',
    orientation_col: str = 'Orientation',
    figsize: tuple = (10, 6),
    show: bool = True,
    save_path: str = None,
    print_df: bool = True
) -> Tuple[plt.Figure, pd.DataFrame]:
    """
    Plots the distribution of orientation values across different labels.
    Colors represent different orientations, not reliability labels.
    
    - Asse X rinominato in "Reliability label".
    - Asse Y rinominato in "N° of articles".
    - Label asse X partono da 1.
    - Ordine Orientation personalizzato.
    - Colori assegnati per Orientation, non per Label.

    Args:
        df (pd.DataFrame): The input DataFrame containing 'label' and 'Orientation' columns.
        label_col (str): The name of the column containing the labels (e.g., 'label').
        orientation_col (str): The name of the column containing the orientation values.
        figsize (tuple): Figure size (width, height) in inches.
        show (bool): If True, displays the plot.
        save_path (str): If provided, saves the plot to this path.
        print_df (bool): If True, prints the aggregated DataFrame to the console output.

    Returns:
        Tuple[plt.Figure, pd.DataFrame]: A tuple containing the Matplotlib Figure object
                                          and the Pandas DataFrame with the aggregated data.
    """

    if label_col not in df.columns:
        raise ValueError(f"'{label_col}' column not found in the DataFrame.")
    if orientation_col not in df.columns:
        raise ValueError(f"'{orientation_col}' column not found in the DataFrame.")

    all_labels = sorted(df[label_col].unique())
    
    # --- Ordine personalizzato delle Orientation ---
    desired_order = [
        "Far Left", "Slightly Left", "Non labelled", "Slightly Right", "Far Right"
    ]
    
    # Ottieni le orientation presenti nel dataframe
    present_orientations = set(df[orientation_col].unique())
    
    # Costruisci la lista ordinata: 
    # 1. Prima quelli nell'ordine desiderato (se esistono nei dati)
    all_orientations = [x for x in desired_order if x in present_orientations]
    
    # 2. Aggiungi eventuali valori presenti nei dati ma non nella lista desiderata (per sicurezza)
    others = [x for x in present_orientations if x not in desired_order]
    all_orientations.extend(sorted(others))
    # ---------------------------------------------------------

    # Define distinct colors for each orientation
    orientation_colors = [
        'tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple',
        'tab:brown', 'tab:pink', 'tab:gray', 'tab:olive', 'tab:cyan'
    ]
    color_map = {ori: orientation_colors[i % len(orientation_colors)] for i, ori in enumerate(all_orientations)}
    
    # Define distinct hatch patterns for accessibility (colorblind, b&w printing)
    hatch_patterns = ['///', '\\\\\\', 'xxx', '---', '+++', '...', 'ooo', '***', '|||', '///\\\\\\']
    
    if len(all_orientations) > len(hatch_patterns):
        print("Warning: Not enough unique hatch patterns for all orientations. Patterns will repeat.")
        hatch_patterns = (hatch_patterns * ((len(all_orientations) // len(hatch_patterns)) + 1))[:len(all_orientations)]
    
    hatch_map = {ori: hatch_patterns[i % len(hatch_patterns)] for i, ori in enumerate(all_orientations)}

    # Prepare data for plotting and for the output DataFrame
    raw_plot_data = [] 
    for label in all_labels:
        subset = df[df[label_col] == label]
        # Reindex using the specific sorted order to ensure correct counting structure
        orientation_counts = subset[orientation_col].value_counts().reindex(all_orientations, fill_value=0)
        
        for ori in all_orientations:
            count = orientation_counts.get(ori, 0)
            raw_plot_data.append({
                'Label': label,
                'Orientation': ori,
                'Count': count
            })
            
    plot_df = pd.DataFrame(raw_plot_data)

    # Optionally print the DataFrame
    if print_df:
        print("\nAggregated Data for Plot:")
        plot_df['Orientation'] = pd.Categorical(plot_df['Orientation'], categories=all_orientations, ordered=True)
        pivot_df = plot_df.pivot_table(index='Orientation', columns='Label', values='Count', fill_value=0)
        print(pivot_df)
        print("-" * 50)

    fig, ax = plt.subplots(figsize=figsize)
    ax.set_axisbelow(True)

    num_orientations = len(all_orientations)
    group_total_width = 0.8 
    bar_width = group_total_width / num_orientations
    
    x_positions = np.arange(len(all_labels))

    # Plot bars with colors based on orientation
    for l_idx, label in enumerate(all_labels):
        subset_data = plot_df.query("Label == @label and Count > 0")
        
        # Filtra mantenendo l'ordine corretto di 'all_orientations'
        present_oris_with_indices = [
            ori for ori in all_orientations if ori in subset_data['Orientation'].values
        ]

        num_present_orientations = len(present_oris_with_indices)

        if num_present_orientations == 0:
            continue

        for k, ori in enumerate(present_oris_with_indices):
            # Calcola il nuovo offset per centrare le barre *presenti*
            offset = (k - num_present_orientations / 2 + 0.5) * bar_width
            
            # Usa il colore e l'hatch basati sull'orientation
            current_color = color_map[ori]
            current_hatch = hatch_map[ori]
            
            height = subset_data[subset_data["Orientation"] == ori]['Count'].values[0]
            
            ax.bar(
                x_positions[l_idx] + offset,
                height,
                width=bar_width,
                color=current_color,
                edgecolor='black',
                linewidth=1.2,
                hatch=current_hatch
            )
            
    # --- Rinomina Assi ---
    ax.set_ylabel("N° of articles", fontsize=14)
    ax.set_xlabel("Reliability label", fontsize=14)
    
    ax.set_xticks(x_positions)
    # --- Etichette X partono da 1 anziché 0 ---
    ax.set_xticklabels([str(i + 1) for i in x_positions], rotation=0, ha='center', fontsize=12, fontweight='bold')
    
    ax.tick_params(axis='y', labelsize=12)

    # Creazione della legenda per le Orientations (colori + hatch patterns)
    orientation_legend_handles = []
    for ori in all_orientations:
        orientation_legend_handles.append(
            Patch(facecolor=color_map[ori], edgecolor='black', linewidth=1.2, hatch=hatch_map[ori], label=str(ori))
        )
        
    orientation_legend = ax.legend(
        handles=orientation_legend_handles,
        title='Leaning',
        loc='upper right',
        frameon=True,
        edgecolor='black',
        fontsize=12,
        title_fontsize=14,
        handlelength=2.0,
        handleheight=2.0
    )

    ax.add_artist(orientation_legend)

    plt.tight_layout()

    if save_path:
        try:
            fig.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"📁 Plot saved to: {save_path}")
        except Exception as e:
            print(f"Error saving plot: {e}")

    if show:
        plt.show()
    
    return fig, plot_df

# Esempio di utilizzo
fig, data = plot_orientation_distribution_per_label(
    df,
    save_path="Results/Plots/orientationDistribution" 
)

/tmp/ipykernel_249192/3443008839.py:101: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_df = plot_df.pivot_table(index='Orientation', columns='Label', values='Count', fill_value=0)



Aggregated Data for Plot:
Label                0      1       2       3      4
Orientation                                         
Far Left          86.0  355.0     6.0   408.0  160.0
Slightly Left      0.0  245.0   174.0  1325.0  663.0
Non labelled      71.0  635.0  3189.0  1964.0  563.0
Slightly Right  4245.0  782.0   960.0   961.0  542.0
Far Right        469.0  999.0   291.0   383.0    0.0
--------------------------------------------------
📁 Plot saved to: Results/Plots/orientationDistribution


/tmp/ipykernel_249192/3443008839.py:189: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()


In [25]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from typing import Tuple
from matplotlib.patches import Patch


plt.rcParams.update({
    "pgf.texsystem": "pdflatex",
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": [],
    "axes.labelsize": 14,
    "font.size": 14,
    "legend.fontsize": 12,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "pgf.rcfonts": False,
})

def plot_reliability_distribution_per_orientation(
    df: pd.DataFrame,
    label_col: str = 'label',
    orientation_col: str = 'Orientation',
    figsize: tuple = (10, 6),
    show: bool = True,
    save_path: str = None,
    print_df: bool = True
) -> Tuple[plt.Figure, pd.DataFrame]:

    # --- Mappa per le fasce di reliability ---
    reliability_ranges = {
        0: r"\textit{l1} (0-39)",
        1: r"\textit{l2} (40-59)",
        2: r"\textit{l3} (60-74)",
        3: r"\textit{l4} (75-99)",
        4: r"\textit{l5} (100)"  # Ho aggiunto 'l' qui per uniformità, o puoi usare r"\textit{5} (100)"
    }

    # --- Controlli ---
    if label_col not in df.columns:
        raise ValueError(f"'{label_col}' column not found in the DataFrame.")
    if orientation_col not in df.columns:
        raise ValueError(f"'{orientation_col}' column not found in the DataFrame.")

    # --- Ordine Orientation ---
    desired_order = [
        "Far Left", "Slightly Left", "Non labelled",
        "Slightly Right", "Far Right"
    ]
    present_orientations = set(df[orientation_col].unique())
    all_orientations = [x for x in desired_order if x in present_orientations]
    all_orientations.extend(sorted(present_orientations - set(all_orientations)))

    # --- Reliability labels ---
    all_labels = sorted(df[label_col].unique())

    # --- Colori + hatch per reputabilità ---
    colors = [
        'tab:blue', 'tab:orange', 'tab:green', 'tab:red',
        'tab:purple', 'tab:brown', 'tab:pink', 'tab:gray'
    ]
    hatches = ['///', '\\\\', 'xxx', '---', '+++', '...', 'ooo', '***']

    color_map = {lab: colors[i % len(colors)] for i, lab in enumerate(all_labels)}
    hatch_map = {lab: hatches[i % len(hatches)] for i, lab in enumerate(all_labels)}

    # --- Aggregazione dati ---
    raw_data = []
    for ori in all_orientations:
        subset = df[df[orientation_col] == ori]
        counts = subset[label_col].value_counts().reindex(all_labels, fill_value=0)

        for lab in all_labels:
            raw_data.append({
                'Orientation': ori,
                'Label': lab,
                'Count': counts[lab]
            })

    plot_df = pd.DataFrame(raw_data)

    # --- Stampa tabella ---
    if print_df:
        print("\nAggregated Data for Plot:")
        pivot_df = plot_df.pivot_table(
            index='Orientation',
            columns='Label',
            values='Count',
            fill_value=0
        )
        # Mostra le label con i range di reliability
        pivot_df.columns = [reliability_ranges.get(c, str(c)) for c in pivot_df.columns]
        print(pivot_df)
        print("-" * 50)

    # --- Plot ---
    fig, ax = plt.subplots(figsize=figsize)

    num_labels = len(all_labels)
    group_width = 0.8
    bar_width = group_width / num_labels
    x_positions = np.arange(len(all_orientations))

    for i, lab in enumerate(all_labels):
        lab_data = plot_df[plot_df['Label'] == lab]
        offset = (i - num_labels / 2 + 0.5) * bar_width

        ax.bar(
            x_positions + offset,
            lab_data['Count'],
            width=bar_width,
            color=color_map[lab],
            edgecolor='black',
            hatch=hatch_map[lab],
            label=reliability_ranges.get(lab, str(lab))  # ← Usa i range di reliability
        )

    # --- Assi ---
    ax.set_xlabel("Leaning")
    ax.set_ylabel("N° of articles")

    ax.set_xticks(x_positions)
    ax.set_xticklabels(
        all_orientations,
        rotation=0,
        ha='center',
    )

    ax.tick_params(axis='y', labelsize=12)

    # --- Legenda ---
    ax.legend(
        title="Reliability label",
        frameon=True,
        edgecolor='black'
    )

    plt.tight_layout()

    ax.grid(axis='y', linestyle='--', alpha=0.5)
    plt.show()

    # --- Salvataggio ---
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"📁 Plot saved to: {save_path}")


    return fig, plot_df

# Esempio di utilizzo
fig, data = plot_reliability_distribution_per_orientation(
    df,
    save_path="Results/Plots/orientationDistributionInverted" 
)


Aggregated Data for Plot:
                \textit{l1} (0-39)  \textit{l2} (40-59)  \textit{l3} (60-74)  \
Orientation                                                                    
Far Left                      86.0                355.0                  6.0   
Far Right                    469.0                999.0                291.0   
Non labelled                  71.0                635.0               3189.0   
Slightly Left                  0.0                245.0                174.0   
Slightly Right              4245.0                782.0                960.0   

                \textit{l4} (75-99)  \textit{l5} (100)  
Orientation                                             
Far Left                      408.0              160.0  
Far Right                     383.0                0.0  
Non labelled                 1964.0              563.0  
Slightly Left                1325.0              663.0  
Slightly Right                961.0              542.0  
-------------

/tmp/ipykernel_249192/222194963.py:142: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()


📁 Plot saved to: Results/Plots/orientationDistributionInverted


In [26]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from typing import Tuple
from matplotlib.colors import LinearSegmentedColormap


def plot_reliability_heatmap_per_orientation(
    df: pd.DataFrame,
    label_col: str = "label",
    orientation_col: str = "Orientation",
    figsize: tuple = (10, 6),
    show: bool = True,
    save_path: str = None,
    print_df: bool = True
) -> Tuple[plt.Figure, pd.DataFrame]:

    # --- Reliability label ranges ---
    reliability_ranges = {
        0: "1 (0–39)",
        1: "2 (40–59)",
        2: "3 (60–74)",
        3: "4 (75–99)",
        4: "5 (100)"
    }

    # --- Checks ---
    if label_col not in df.columns:
        raise ValueError(f"'{label_col}' column not found in the DataFrame.")
    if orientation_col not in df.columns:
        raise ValueError(f"'{orientation_col}' column not found in the DataFrame.")

    # --- Orientation order ---
    desired_order = [
        "Far Left", "Slightly Left", "Non labelled",
        "Slightly Right", "Far Right"
    ]
    present_orientations = set(df[orientation_col].unique())
    all_orientations = [x for x in desired_order if x in present_orientations]
    all_orientations.extend(sorted(present_orientations - set(all_orientations)))

    # --- Reliability labels (x-axis) ---
    all_labels = sorted(df[label_col].unique())

    # --- Aggregation ---
    heatmap_df = (
        df
        .groupby([orientation_col, label_col])
        .size()
        .unstack(fill_value=0)
        .reindex(index=all_orientations, columns=all_labels, fill_value=0)
    )

    # Rename columns using reliability ranges
    heatmap_df.columns = [reliability_ranges.get(c, str(c)) for c in heatmap_df.columns]

    # --- Print table ---
    if print_df:
        print("\nAggregated Data for Heatmap:")
        print(heatmap_df)
        print("-" * 50)

    # --- Custom blue colormap ---
    colors = [
        "#f7fbff", "#deebf7", "#c6dbef", "#9ecae1",
        "#6baed6", "#4292c6", "#2171b5", "#08519c", "#08306b"
    ]
    cmap = LinearSegmentedColormap.from_list("custom_blues", colors)

    # --- Plot ---
    fig, ax = plt.subplots(figsize=figsize)
    ax.grid(False)

    im = ax.imshow(
        heatmap_df.values,
        cmap=cmap,
        aspect="auto"
    )

    # --- Axes ---
    ax.set_xticks(np.arange(len(heatmap_df.columns)))
    ax.set_xticklabels(heatmap_df.columns, fontsize=12, fontweight="bold")

    ax.set_yticks(np.arange(len(heatmap_df.index)))
    ax.set_yticklabels(heatmap_df.index, fontsize=12, fontweight="bold")

    ax.set_xlabel("Reliability level", fontsize=14)
    ax.set_ylabel("Political leaning", fontsize=14)

    # --- Cell annotations ---
    for i in range(heatmap_df.shape[0]):
        for j in range(heatmap_df.shape[1]):
            value = heatmap_df.iloc[i, j]
            ax.text(
                j, i, value,
                ha="center", va="center",
                fontsize=11,
                color="black" if value < heatmap_df.values.max() * 0.6 else "white"
            )

    # --- Colorbar ---
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("N° of articles", fontsize=12)
    cbar.ax.tick_params(labelsize=11)

    plt.tight_layout()

    if show:
        plt.show()

    # --- Save ---
    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"📁 Heatmap saved to: {save_path}")

    return fig, heatmap_df


# Example usage
fig, data = plot_reliability_heatmap_per_orientation(
    df,
    save_path="Results/Plots/orientationReliabilityHeatmap"
)



Aggregated Data for Heatmap:
                1 (0–39)  2 (40–59)  3 (60–74)  4 (75–99)  5 (100)
Orientation                                                       
Far Left              86        355          6        408      160
Slightly Left          0        245        174       1325      663
Non labelled          71        635       3189       1964      563
Slightly Right      4245        782        960        961      542
Far Right            469        999        291        383        0
--------------------------------------------------


/tmp/ipykernel_249192/2230547229.py:109: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()


📁 Heatmap saved to: Results/Plots/orientationReliabilityHeatmap


# Analisi statistica ChatGPT Marinella

In [27]:
from scipy.stats import chi2_contingency
import pandas as pd
import numpy as np

def calculate_cramers_v(x, y):
    """
    Calcola il Cramér's V per misurare l'associazione tra due variabili categoriche.
    
    Cramér's V è una misura simmetrica dell'associazione tra due variabili nominali.
    Range: [0, 1] dove 0 = nessuna associazione, 1 = associazione perfetta
    
    Args:
        x: Prima variabile (array-like)
        y: Seconda variabile (array-like)
    
    Returns:
        float: Valore di Cramér's V
    """
    # Crea tabella di contingenza
    confusion_matrix = pd.crosstab(x, y)
    
    # Esegui test chi-quadrato
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    
    # Calcola min(r-1, c-1) dove r=righe, c=colonne
    min_dim = min(confusion_matrix.shape[0] - 1, confusion_matrix.shape[1] - 1)
    
    # Cramér's V = sqrt(chi2 / (n * min_dim))
    cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0
    
    return cramers_v

def analyze_error_association(
    dfs, model_names,
    col_orientation="Orientation",
    col_label="label",
    col_pred="predicted_label"
):
    """
    Calcola il test chi-quadrato (p-value) e Cramér's V per l'associazione tra
    orientamento e occorrenza di errori per ciascun modello.
    
    Il test chi-quadrato verifica se esiste una relazione statistica significativa
    tra l'orientamento e la presenza di errori.
    
    H0: Non c'è associazione tra orientamento e errori
    H1: C'è associazione tra orientamento e errori
    
    Args:
        dfs: Lista di DataFrame (uno per modello) o singolo DataFrame
        model_names: Lista di nomi dei modelli o singolo nome
        col_orientation: Nome colonna orientamento
        col_label: Nome colonna etichetta vera
        col_pred: Nome colonna etichetta predetta
    
    Returns:
        pd.DataFrame: Tabella con statistiche per ogni modello
    """
    
    print("="*100)
    print("ANALISI ASSOCIAZIONE TRA ORIENTAMENTO E ERRORI")
    print("(Test Chi-Quadrato e Cramér's V)")
    print("="*100)
    
    # --- PREPARAZIONE INPUT ---
    if not isinstance(dfs, list):
        dfs = [dfs]
    if not isinstance(model_names, list):
        model_names = [model_names]
    
    if len(dfs) != len(model_names):
        raise ValueError(f"Numero di DataFrame ({len(dfs)}) != numero di modelli ({len(model_names)})")
    
    print(f"\n✓ Analizzando {len(model_names)} modelli: {model_names}")
    print(f"✓ Colonna orientamento: '{col_orientation}'")
    print(f"✓ Ipotesi nulla (H0): Non c'è associazione tra orientamento e errori")
    print(f"✓ Significatività: α = 0.05")
    
    # --- CALCOLO STATISTICHE PER OGNI MODELLO ---
    results = []
    
    for idx, (df, model_name) in enumerate(zip(dfs, model_names)):
        print(f"\n" + "─"*100)
        print(f"MODELLO {idx+1}/{len(model_names)}: {model_name}")
        print("─"*100)
        
        temp_df = df.copy()
        
        # Crea colonna isError se non esiste
        if "isError" not in temp_df.columns:
            temp_df["isError"] = (temp_df[col_label] != temp_df[col_pred]).astype(int)
        else:
            temp_df["isError"] = temp_df["isError"].astype(int)
        
        # Sostituisci "Non labelled" con "Non labelled"
        temp_df[col_orientation] = temp_df[col_orientation].replace("Non labelled", "Non labelled")
        
        # --- TABELLA DI CONTINGENZA ---
        print(f"\n📊 Tabella di contingenza (Orientamento vs Errori):\n")
        contingency = pd.crosstab(
            temp_df[col_orientation],
            temp_df["isError"],
            margins=True,
            margins_name="TOTALE"
        )
        contingency.columns = ["Corretto (0)", "Errato (1)", "TOTALE"]
        print(contingency)
        
        # --- TEST CHI-QUADRATO ---
        print(f"\n🧪 Test Chi-Quadrato:")
        print("   " + "─"*70)
        
        # Crea tabella senza margini per il test
        contingency_no_margins = pd.crosstab(
            temp_df[col_orientation],
            temp_df["isError"]
        )
        
        chi2, p_value, dof, expected_freq = chi2_contingency(contingency_no_margins)
        
        print(f"   χ² (chi-quadrato): {chi2:.4f}")
        print(f"   p-value: {p_value:.6f}")
        print(f"   Gradi di libertà: {dof}")
        
        # Interpretazione
        if p_value < 0.05:
            sig_text = "✓ SIGNIFICATIVO (p < 0.05)"
            interpretation = "C'è una relazione statisticamente significativa tra orientamento e errori"
        else:
            sig_text = "✗ NON SIGNIFICATIVO (p ≥ 0.05)"
            interpretation = "Non c'è evidenza di relazione tra orientamento e errori"
        
        print(f"\n   Risultato: {sig_text}")
        print(f"   Interpretazione: {interpretation}")
        
        # --- CRAMÉR'S V ---
        print(f"\n📈 Cramér's V (Misura dell'associazione):")
        print("   " + "─"*70)
        
        cramers_v = calculate_cramers_v(temp_df[col_orientation], temp_df["isError"])
        
        print(f"   V = {cramers_v:.4f}")
        
        # Interpretazione Cramér's V (scala di Cohen)
        # Per una tabella 5x2 (5 orientamenti, 2 esiti: corretto/errato)
        if cramers_v < 0.1:
            effect_size = "Negligibile"
        elif cramers_v < 0.3:
            effect_size = "Piccolo"
        elif cramers_v < 0.5:
            effect_size = "Medio"
        else:
            effect_size = "Grande"
        
        print(f"   Interpretazione: {effect_size} (effect size)")
        print(f"   • V < 0.1: Negligibile")
        print(f"   • 0.1 ≤ V < 0.3: Piccolo")
        print(f"   • 0.3 ≤ V < 0.5: Medio")
        print(f"   • V ≥ 0.5: Grande")
        
        # --- STATISTICHE DESCRITTIVE ---
        print(f"\n📊 Statistiche descrittive:")
        print("   " + "─"*70)
        
        n_total = len(temp_df)
        n_errors = temp_df["isError"].sum()
        error_rate = n_errors / n_total * 100
        
        print(f"   Campioni totali: {n_total}")
        print(f"   Predizioni errate: {n_errors} ({error_rate:.2f}%)")
        print(f"   Predizioni corrette: {n_total - n_errors} ({100-error_rate:.2f}%)")
        
        # Error rate per orientamento
        print(f"\n   Error rate per orientamento:")
        orient_stats = temp_df.groupby(col_orientation)["isError"].agg(['sum', 'count', 'mean'])
        orient_stats.columns = ['Errori', 'Totale', 'Error Rate']
        orient_stats['Error Rate %'] = orient_stats['Error Rate'] * 100
        orient_stats = orient_stats[['Totale', 'Errori', 'Error Rate %']].round(2)
        
        for orient in orient_stats.index:
            row = orient_stats.loc[orient]
            print(f"      {orient:<20}: {int(row['Errori'])}/{int(row['Totale'])} ({row['Error Rate %']:.1f}%)")
        
        # --- FREQUENZE ATTESE ---
        print(f"\n📋 Frequenze attese (per il test chi-quadrato):")
        print("   " + "─"*70)
        
        expected_df = pd.DataFrame(
            expected_freq,
            index=contingency_no_margins.index,
            columns=["Attese (Corretto)", "Attese (Errato)"]
        ).round(2)
        print(expected_df)
        
        # Verifica assunzioni chi-quadrato
        print(f"\n⚠️  Assunzioni test chi-quadrato:")
        min_expected = expected_freq.min()
        cells_below_5 = (expected_freq < 5).sum()
        total_cells = expected_freq.size
        pct_below_5 = cells_below_5 / total_cells * 100
        
        print(f"   • Min frequenza attesa: {min_expected:.2f}")
        print(f"   • Cell con freq attesa < 5: {cells_below_5}/{total_cells} ({pct_below_5:.1f}%)")
        
        if min_expected < 1:
            print(f"   ⚠️  ATTENZIONE: Alcune frequenze attese sono < 1!")
            print(f"      Il test chi-quadrato potrebbe non essere affidabile.")
        elif pct_below_5 > 20:
            print(f"   ⚠️  ATTENZIONE: Più del 20% delle cell hanno freq attesa < 5!")
            print(f"      Considera di combinare categorie o usare test esatti (Fisher).")
        else:
            print(f"   ✓ Assunzioni soddisfatte")
        
        # --- SALVA RISULTATI ---
        results.append({
            'Modello': model_name,
            'Chi-Quadrato': chi2,
            'p-value': p_value,
            'Cramér V': cramers_v,
            'Significativo (α=0.05)': 'Sì' if p_value < 0.05 else 'No',
            'Effect Size': effect_size,
            'Campioni': n_total,
            'Errori': n_errors,
            'Error Rate %': error_rate
        })
    
    # --- TABELLA RIEPILOGATIVA FINALE ---
    print("\n" + "="*100)
    print("TABELLA RIEPILOGATIVA")
    print("="*100 + "\n")
    
    results_df = pd.DataFrame(results)
    
    # Formatta per la visualizzazione
    display_df = results_df.copy()
    display_df['Chi-Quadrato'] = display_df['Chi-Quadrato'].apply(lambda x: f"{x:.4f}")
    display_df['p-value'] = display_df['p-value'].apply(lambda x: f"{x:.6f}")
    display_df['Cramér V'] = display_df['Cramér V'].apply(lambda x: f"{x:.4f}")
    display_df['Error Rate %'] = display_df['Error Rate %'].apply(lambda x: f"{x:.2f}%")
    
    print(display_df.to_string(index=False))
    
    print("\n" + "="*100)
    print("INTERPRETAZIONE GENERALE")
    print("="*100)
    
    print("\n📌 Cramér's V indica la FORZA dell'associazione:")
    print("   • Valori vicini a 0: Orientamento e errori sono indipendenti")
    print("   • Valori vicini a 1: Orientamento e errori sono fortemente associati")
    print("     (certi orientamenti hanno tassi di errore significativamente diversi)")
    
    print("\n📌 p-value indica la SIGNIFICATIVITÀ STATISTICA:")
    print("   • p < 0.05: La relazione è statistically significant (rifiuta H0)")
    print("   • p ≥ 0.05: La relazione potrebbe essere dovuta al caso (accetta H0)")
    
    print("\n" + "="*100)
    
    return results_df


# UTILIZZO:
results = analyze_error_association(
    [bertErrors, llamaErrors, robertaErrors, mistralErrors],
    ["BERT", "Llama", "Roberta", "Mistral"],
    col_orientation="Orientation",
    col_label="label",
    col_pred="predicted_label"
)

ANALISI ASSOCIAZIONE TRA ORIENTAMENTO E ERRORI
(Test Chi-Quadrato e Cramér's V)

✓ Analizzando 4 modelli: ['BERT', 'Llama', 'Roberta', 'Mistral']
✓ Colonna orientamento: 'Orientation'
✓ Ipotesi nulla (H0): Non c'è associazione tra orientamento e errori
✓ Significatività: α = 0.05

────────────────────────────────────────────────────────────────────────────────────────────────────
MODELLO 1/4: BERT
────────────────────────────────────────────────────────────────────────────────────────────────────

📊 Tabella di contingenza (Orientamento vs Errori):

                Corretto (0)  Errato (1)  TOTALE
Orientation                                     
Far Left                 299         716    1015
Far Right                678        1464    2142
Non labelled            4292        2130    6422
Slightly Left           1018        1389    2407
Slightly Right          5062        2428    7490
TOTALE                 11349        8127   19476

🧪 Test Chi-Quadrato:
   ────────────────────────────

## Grafici per results di reliability come in political leaning

In [28]:
"""
Generate Mean Error Direction plot for Reliability Labels.

This script computes the mean signed error (predicted - true label) for 
misclassified instances, stratified by TRUE reliability label.

Positive values indicate overestimation (model predicts higher reliability)
Negative values indicate underestimation (model predicts lower reliability)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Read the confusion matrix data
df = pd.read_csv('Results/Classification/confusion_matrices_all_models.csv')

# Aggregate across all 5 folds for each model
# Sum the values for each (model, true_label, predicted_label) combination
df_agg = df.groupby(['model', 'true_label', 'predicted_label'])['value'].sum().reset_index()

# Models to process
models = ['bert', 'llama3', 'roberta']
model_labels = ['BERT', 'LLaMA 3', 'RoBERTa']
reliability_labels = [1, 2, 3, 4, 5]

# Dictionary to store mean signed errors
results = {model: {} for model in models}

for model in models:
    model_data = df_agg[df_agg['model'] == model]
    
    for true_label in reliability_labels:
        # Filter data for this true label
        label_data = model_data[model_data['true_label'] == true_label]
        
        # Calculate signed error for misclassified instances only
        total_signed_error = 0
        total_misclassifications = 0
        
        for _, row in label_data.iterrows():
            pred_label = row['predicted_label']
            count = row['value']
            
            # Only consider misclassifications (predicted != true)
            if pred_label != true_label:
                signed_error = pred_label - true_label
                total_signed_error += signed_error * count
                total_misclassifications += count
        
        # Calculate mean signed error
        if total_misclassifications > 0:
            mean_signed_error = total_signed_error / total_misclassifications
        else:
            mean_signed_error = 0
        
        results[model][true_label] = mean_signed_error

# Print the results
print("Mean Signed Error Direction by Reliability Label")
print("=" * 60)
print(f"{'Label':<10} {'BERT':>12} {'LLaMA 3':>12} {'RoBERTa':>12}")
print("-" * 60)
for label in reliability_labels:
    print(f"{label:<10} {results['bert'][label]:>12.3f} {results['llama3'][label]:>12.3f} {results['roberta'][label]:>12.3f}")

# Create the plot
fig, ax = plt.subplots(figsize=(10, 6))

# Set up bar positions
x = np.arange(len(reliability_labels))
width = 0.25  # width of bars

# Colors for each model (similar to the original plot style)
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # blue, orange, green

# Create bars for each model
bars1 = ax.bar(x - width, [results['bert'][l] for l in reliability_labels], 
               width, label='BERT', color=colors[0], edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x, [results['llama3'][l] for l in reliability_labels], 
               width, label='LLaMA 3', color=colors[1], edgecolor='black', linewidth=0.5)
bars3 = ax.bar(x + width, [results['roberta'][l] for l in reliability_labels], 
               width, label='RoBERTa', color=colors[2], edgecolor='black', linewidth=0.5)

# Add horizontal line at y=0
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.8)

# Customize the plot
ax.set_xlabel('True Reliability Label', fontsize=12)
ax.set_ylabel('Mean Error Direction', fontsize=12)
ax.set_title('Mean Error Direction by Reliability Label', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(reliability_labels)
ax.legend(loc='best')

# Add grid for better readability
ax.yaxis.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)

# Adjust layout
plt.tight_layout()

# Save the figure
plt.savefig('Results/PlotserrorsDirection_reliability.pdf', format='pdf', dpi=300, bbox_inches='tight')

print("\n" + "=" * 60)
print("Plot saved as:")
print("  - errorsDirection_reliability.pdf")

# Also print some interpretation
print("\n" + "=" * 60)
print("INTERPRETATION:")
print("-" * 60)
print("Positive values: Model OVERESTIMATES reliability (predicts higher than true)")
print("Negative values: Model UNDERESTIMATES reliability (predicts lower than true)")
print("\nFor example:")
for label in reliability_labels:
    avg_error = np.mean([results[m][label] for m in models])
    if avg_error > 0.1:
        print(f"  Label {label}: Models tend to OVERESTIMATE reliability (avg: {avg_error:.2f})")
    elif avg_error < -0.1:
        print(f"  Label {label}: Models tend to UNDERESTIMATE reliability (avg: {avg_error:.2f})")
    else:
        print(f"  Label {label}: Models are relatively balanced (avg: {avg_error:.2f})")

plt.show()


Mean Signed Error Direction by Reliability Label
Label              BERT      LLaMA 3      RoBERTa
------------------------------------------------------------
1                 2.297        2.229        2.042
2                 1.531        1.415        1.653
3                 0.346        0.099        0.132
4                -0.828       -0.865       -0.591
5                -1.598       -1.720       -1.673

Plot saved as:
  - errorsDirection_reliability.pdf

INTERPRETATION:
------------------------------------------------------------
Positive values: Model OVERESTIMATES reliability (predicts higher than true)
Negative values: Model UNDERESTIMATES reliability (predicts lower than true)

For example:
  Label 1: Models tend to OVERESTIMATE reliability (avg: 2.19)
  Label 2: Models tend to OVERESTIMATE reliability (avg: 1.53)
  Label 3: Models tend to OVERESTIMATE reliability (avg: 0.19)
  Label 4: Models tend to UNDERESTIMATE reliability (avg: -0.76)
  Label 5: Models tend to UNDERESTIMAT

/tmp/ipykernel_249192/72636177.py:126: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()


In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ----------------------------
# 1. Read data
# ----------------------------
df = pd.read_csv('Results/Classification/confusion_matrices_all_models.csv')
df_agg = df.groupby(['model', 'true_label', 'predicted_label'])['value'].sum().reset_index()

# ----------------------------
# 2. Define models and labels
# ----------------------------
models = ['bert', 'llama3', 'roberta']
reliability_labels = [1, 2, 3, 4, 5]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

# ----------------------------
# 3. Prepare boxplot data
# ----------------------------
boxplot_data = {model: {label: [] for label in reliability_labels} for model in models}

for model in models:
    model_data = df_agg[df_agg['model'] == model]
    for _, row in model_data.iterrows():
        true_label = row['true_label']
        pred_label = row['predicted_label']
        count = row['value']
        if pred_label != true_label:
            error = pred_label - true_label
            boxplot_data[model][true_label].extend([error]*count)

# ----------------------------
# 4. Plot grouped compact boxplots
# ----------------------------
fig, ax = plt.subplots(figsize=(10,6))

positions = np.arange(len(reliability_labels))  # 0,1,2,3,4
width = 0.2  # small width for each boxplot
offsets = [-width, 0, width]  # shift boxes for each model

for i, model in enumerate(models):
    data = [boxplot_data[model][label] for label in reliability_labels]
    bp = ax.boxplot(
        data,
        positions=positions + offsets[i],
        widths=width,
        patch_artist=True,
        boxprops=dict(facecolor=colors[i], edgecolor='black'),
        medianprops=dict(color='black'),
        whiskerprops=dict(color='black'),
        capprops=dict(color='black'),
        flierprops=dict(marker='o', markersize=2, alpha=0.5),
        manage_ticks=False
    )

# horizontal line at zero
ax.axhline(0, color='black', linewidth=0.8)

# ticks and labels
ax.set_xticks(positions)
ax.set_xticklabels(reliability_labels)
ax.set_xlabel('True Reliability Label')
ax.set_ylabel('Signed Error (Predicted − True)')
ax.set_title('Error Direction by Reliability Label (Compact Grouped Boxplots)')

# legend
legend_elements = [Patch(facecolor=colors[i], label=model.capitalize()) for i, model in enumerate(models)]
ax.legend(handles=legend_elements, loc='best')

ax.yaxis.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)
plt.tight_layout()

# save figure
plt.savefig('Results/Plots/errorsDirection_reliability_compact', dpi=300, bbox_inches='tight')
plt.show()


/tmp/ipykernel_249192/472306422.py:78: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()


In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ----------------------------
# 1. Read data & Data Setup
# ----------------------------
# Assumiamo che df sia già caricato come nel tuo esempio.
# Se carichi da file, decommenta la riga sotto:
# df = pd.read_csv('Results/Classification/confusion_matrices_all_models.csv')

# Aggreghiamo i dati nel caso ci siano più fold, sommando i 'value'
df_agg = df.groupby(['model', 'true_label', 'predicted_label'])['value'].sum().reset_index()

# ----------------------------
# 2. Define models and labels
# ----------------------------
models = ['bert', 'llama3', 'roberta']
reliability_labels = [1, 2, 3, 4, 5]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

# ----------------------------
# 3. Calculate Mean and Error Bars (NESSUNA MODIFICA QUI)
# ----------------------------
plot_data = {model: {'means': [], 'errors': []} for model in models}

for model in models:
    for t_label in reliability_labels:
        subset = df_agg[(df_agg['model'] == model) & (df_agg['true_label'] == t_label)]
        
        if subset.empty:
            plot_data[model]['means'].append(0)
            plot_data[model]['errors'].append(0)
            continue
            
        # Calcoliamo l'errore (Predicted - True)
        errors = subset['predicted_label'] - subset['true_label']
        counts = subset['value']
        
        # Calcolo Media Pesata
        total_count = counts.sum()
        weighted_mean = (errors * counts).sum() / total_count
        
        # Calcolo Deviazione Standard e Errore Standard (SEM)
        variance = ((errors - weighted_mean)**2 * counts).sum() / total_count
        std_dev = np.sqrt(variance)
        sem = std_dev / np.sqrt(total_count) 
        
        # Usiamo Intervallo di Confidenza al 95% (1.96 * SEM) per la barra di errore
        error_margin = 1.96 * sem 
        
        plot_data[model]['means'].append(weighted_mean)
        plot_data[model]['errors'].append(error_margin)

# ----------------------------
# 4. Plot Bar Chart with Error Bars (MODIFICATO)
# ----------------------------
fig, ax = plt.subplots(figsize=(10, 6))

positions = np.arange(len(reliability_labels))  # 0,1,2,3,4
width = 0.25  # Spessore delle barre (leggermente più larghe)
offsets = [-width, 0, width]  # shift per ogni modello

for i, model in enumerate(models):
    means = plot_data[model]['means']
    yerrs = plot_data[model]['errors']
    
    # --- MODIFICA PRINCIPALE: Uso di ax.bar ---
    ax.bar(
        positions + offsets[i], # Posizione X
        means,                  # Altezza barra (La Media)
        width=width,            # Larghezza barra
        yerr=yerrs,             # La barra di errore
        capsize=5,              # Cappuccio orizzontale sulla barra errore
        color=colors[i],        # Colore riempimento
        edgecolor='black',      # Bordo nero per definizione
        linewidth=1,
        label=model.capitalize(),
        # Opzioni specifiche per lo stile delle error bars (colore nero, spessore)
        error_kw=dict(ecolor='black', elinewidth=1.5) 
    )

# Linea orizzontale a zero (nessun errore) ben visibile
ax.axhline(0, color='black', linewidth=1.2, linestyle='-')

# Griglia e formattazione
ax.set_xticks(positions)
ax.set_xticklabels(reliability_labels)
ax.set_xlabel('True Reliability Label', fontsize=12)
ax.set_ylabel('Mean Signed Error (Predicted − True)', fontsize=12)
# Titolo aggiornato per riflettere che sono barre e non boxplot
ax.set_title('Mean Error Direction by Reliability Label (Bar Chart with 95% CI)', fontsize=14)

# Legenda
ax.legend(loc='best', title='Models')

# Griglia solo orizzontale e messa dietro le barre
ax.yaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('Results/Plots/errorsDirection_reliability_barplot.pdf', dpi=300, bbox_inches='tight')
plt.show()

/tmp/ipykernel_249192/3176247322.py:104: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()


In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------
# 1. Setup Dati (Simulazione o Caricamento)
# ----------------------------
# df = pd.read_csv('Results/Classification/confusion_matrices_all_models.csv')
# Se df è già caricato, usiamo quello:
df_agg = df.groupby(['model', 'true_label', 'predicted_label'])['value'].sum().reset_index()

# ----------------------------
# 2. Definizioni Modelli, Colori e PATTERN
# ----------------------------
models = ['bert', 'llama3', 'roberta']
reliability_labels = [1, 2, 3, 4, 5]

# Colori simili all'immagine (Blu, Arancio, Verde)
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

# QUI DEFINIAMO I TRATTEGGI (Hatching)
# '/' = linee diagonali, '\\' = diagonali inverse, 'x' = incrocio, '.' = punti, etc.
# Ripeterli aumenta la densità (es. '//' è più fitto di '/')
patterns = ['//', '\\\\', '||'] 

# ----------------------------
# 3. Calcoli Statistici (Media ed Errore)
# ----------------------------
plot_data = {model: {'means': [], 'errors': []} for model in models}

for model in models:
    for t_label in reliability_labels:
        subset = df_agg[(df_agg['model'] == model) & (df_agg['true_label'] == t_label)]
        if subset.empty:
            plot_data[model]['means'].append(0)
            plot_data[model]['errors'].append(0)
            continue
            
        errors = subset['predicted_label'] - subset['true_label']
        counts = subset['value']
        
        total_count = counts.sum()
        weighted_mean = (errors * counts).sum() / total_count
        variance = ((errors - weighted_mean)**2 * counts).sum() / total_count
        std_dev = np.sqrt(variance)
        sem = std_dev / np.sqrt(total_count) 
        error_margin = 1.96 * sem # 95% CI
        
        plot_data[model]['means'].append(weighted_mean)
        plot_data[model]['errors'].append(error_margin)

# ----------------------------
# 4. Plot con Stile "Hatching"
# ----------------------------
fig, ax = plt.subplots(figsize=(10, 6))

positions = np.arange(len(reliability_labels))
width = 0.25
offsets = [-width, 0, width]

for i, model in enumerate(models):
    means = plot_data[model]['means']
    yerrs = plot_data[model]['errors']
    
    # Disegna le barre con il parametro 'hatch'
    bars = ax.bar(
        positions + offsets[i], 
        means, 
        width=width, 
        yerr=yerrs, 
        capsize=4, 
        
        # Stile Colore + Texture
        color=colors[i],      # Colore di fondo
        edgecolor='black',    # Colore del bordo e del tratteggio
        hatch=patterns[i],    # IL TRUCCO PER L'EFFETTO GRAFICO
        alpha=0.9,            # Leggera trasparenza per non rendere il nero troppo pesante
        
        label=model.capitalize(),
        error_kw=dict(ecolor='black', elinewidth=1.5)
    )

# Linea dello zero
ax.axhline(0, color='black', linewidth=1, linestyle='-', alpha=0.8)

# Griglia tratteggiata leggera sullo sfondo
ax.yaxis.grid(True, linestyle='--', alpha=0.5, color='gray')
ax.set_axisbelow(True)

# Etichette
ax.set_xticks(positions)
ax.set_xticklabels(reliability_labels, fontsize=11)
ax.set_xlabel('True Reliability Label', fontsize=12, labelpad=10)
ax.set_ylabel('Mean Signed Error', fontsize=12)
ax.set_title('Mean Error Direction (Textured Bar Chart)', fontsize=14)

# Legenda personalizzata
# Matplotlib gestisce automaticamente i pattern nella legenda se usiamo le barre
ax.legend(title='Models', fontsize=11, loc='upper left', framealpha=1, edgecolor='black')

plt.tight_layout()
plt.savefig('Results/Plots/errorsDirection_textured', dpi=300, bbox_inches='tight')
plt.show()

/tmp/ipykernel_249192/481305861.py:103: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()


In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ----------------------------
# 1. Read data
# ----------------------------
# Assumiamo che il file sia presente, altrimenti decommenta la riga sotto
# df = pd.read_csv('Results/Classification/confusion_matrices_all_models.csv')
# Simulazione aggregazione (assicurati di avere 'df' caricato)
df_agg = df.groupby(['model', 'true_label', 'predicted_label'])['value'].sum().reset_index()

# ----------------------------
# 2. Define models, labels, colors AND PATTERNS
# ----------------------------
models = ['bert', 'llama3', 'roberta']
reliability_labels = [1, 2, 3, 4, 5]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

# Definiamo i pattern per le texture:
# '/' = diagonale, '\\' = diagonale inversa, 'x' = incrocio (o puoi usare '.', '+', ecc.)
patterns = ['//', '\\\\', 'xx'] 

# ----------------------------
# 3. Prepare data for mean and error bars
# ----------------------------
mean_errors = {model: {} for model in models}
std_errors = {model: {} for model in models}

for model in models:
    model_data = df_agg[df_agg['model'] == model]
    for label in reliability_labels:
        errors = []
        for _, row in model_data.iterrows():
            true_label = row['true_label']
            pred_label = row['predicted_label']
            count = row['value']
            
            # NOTA: Qui stai calcolando l'errore SOLO quando la predizione è diversa dal vero
            if true_label == label and pred_label != true_label:
                error = pred_label - true_label
                errors.extend([error] * count)
        
        # Se non ci sono errori nella lista (o tutti corretti), mettiamo 0
        if not errors:
            errors = [0]
        
        mean_errors[model][label] = np.mean(errors)
        std_errors[model][label] = np.std(errors)

# ----------------------------
# 4. Plot grouped bar charts with error bars AND TEXTURE
# ----------------------------
fig, ax = plt.subplots(figsize=(12, 6))

positions = np.arange(len(reliability_labels))
width = 0.25
offsets = [-width, 0, width]

for i, model in enumerate(models):
    means = [mean_errors[model][label] for label in reliability_labels]
    stds = [std_errors[model][label] for label in reliability_labels]
    
    ax.bar(
        positions + offsets[i],
        means,
        width,
        label=model.capitalize(),
        color=colors[i],
        edgecolor='black',  # Colore del contorno e della texture
        linewidth=0.8,
        hatch=patterns[i],  # <--- AGGIUNTA TEXTURE QUI
        yerr=stds,
        capsize=5,
        error_kw={'elinewidth': 1.5, 'capthick': 2, 'ecolor': 'black'}
    )

# Horizontal line at zero
ax.axhline(0, color='black', linewidth=1)

# Ticks and labels
ax.set_xticks(positions)
ax.set_xticklabels(reliability_labels)
ax.set_xlabel('True Reliability Label', fontsize=11)
ax.set_ylabel('Mean Signed Error (Predicted − True)', fontsize=11)
ax.set_title('Mean Error Direction by Reliability Label with Standard Deviation', fontsize=12)

# Legend (Aggiornata per includere la texture)
legend_elements = [
    Patch(facecolor=colors[i], hatch=patterns[i], edgecolor='black', label=model.capitalize()) 
    for i, model in enumerate(models)
]
ax.legend(handles=legend_elements, loc='upper right') # Spostata in alto a sinistra per visibilità

ax.yaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('Results/Plots/errorsDirection_reliability_textured', dpi=300, bbox_inches='tight')
plt.show()

/tmp/ipykernel_249192/3720753282.py:101: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()


In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ast
import matplotlib

# --- Configurazione Stile (LaTeX / Serif) ---
plt.rcParams.update({
    "pgf.texsystem": "pdflatex",
    "text.usetex": True,
    "font.family": "serif",
    "axes.labelsize": 14,
    "font.size": 14,
    "legend.fontsize": 11,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
})


labelMapping = {
    0: r"\textit{l1} (0-39)",
    1: r"\textit{l2} (40-59)",
    2: r"\textit{l3} (60-74)",
    3: r"\textit{l4} (75-99)",
    4: r"\textit{l5} (100)"  
}


# --- Funzione per calcolare l'Error Rate dai DataFrame ---
def calculate_class_error_rates(df, model_name):
    """
    Estrae le matrici di confusione (sommandole tra i fold)
    e calcola l'error rate per ogni classe (0-4).
    Aggiunge stampa testuale dei risultati per classe.
    """
    total_cm = np.zeros((5, 5), dtype=int)

    for matrix_str in df['Confusion_Matrix']:
        cm = np.array(ast.literal_eval(matrix_str))
        total_cm += cm

    # Totale veri campioni per classe
    row_sums = total_cm.sum(axis=1)
    correct_preds = total_cm.diagonal()

    # Accuracy e Error Rate per classe
    class_accuracies = np.divide(
        correct_preds,
        row_sums,
        out=np.zeros_like(correct_preds, dtype=float),
        where=row_sums != 0
    )
    error_rates = 1 - class_accuracies

    # ── STAMPA TESTUALE ───────────────────────────────────────────────
    print(f"\n=== Error Rates per modello: {model_name} ===")
    print("Classe          | Error Rate | Corrette / Totali")
    print("-" * 50)
    
    for i, (err, correct, total) in enumerate(zip(error_rates, correct_preds, row_sums)):
        label = labelMapping[i]
        if total > 0:
            print(f"{label:<15} | {err:>9.3%}  | {correct:>4d} / {total:>4d}")
        else:
            print(f"{label:<15} |      N/A    |    0 / 0 (nessun esempio)")
    
    print("-" * 50)
    # ──────────────────────────────────────────────────────────────────

    return error_rates

# --- CARICAMENTO DATI ---
# (Assicurati che i DataFrame bertResults, llamaResults, robertaResults siano definiti)
# Esempio:
# bertResults = pd.read_csv('Results/Classification/bert.csv')
# llamaResults = pd.read_csv('Results/Classification/llama.csv')
# robertaResults = pd.read_csv('Results/Classification/roberta.csv')

# --- ESTRAZIONE DATI con Nomi Aggiornati ---
# Nota: L'ordine qui non influenza il grafico, ma solo la stampa in console
bert_errors    = calculate_class_error_rates(bertResults,    "BERT")
roberta_errors = calculate_class_error_rates(robertaResults, "RoBERTa")
llama_errors   = calculate_class_error_rates(llamaResults,   "Llama")

# --- PLOTTING ---
labels = [labelMapping[i] for i in range(5)]
x = np.arange(len(labels))
width = 0.25  # Larghezza delle barre

fig, ax = plt.subplots(figsize=(10, 6))

# 1. BERT (Sinistra) -> Pattern '//'
ax.bar(
    x - width, bert_errors, width, label='BERT',
    color='tab:blue', hatch='//', edgecolor='black', zorder=3
)

# 2. RoBERTa (Centro) -> Pattern '\\' (scritto '\\\\' in python)
# Nota: Spostato al centro (x)
ax.bar(
    x, roberta_errors, width, label='RoBERTa',
    color='tab:orange', hatch='\\\\', edgecolor='black', zorder=3
)

# 3. Llama (Destra) -> Pattern 'xx'
# Nota: Spostato a destra (x + width)
ax.bar(
    x + width, llama_errors, width, label='Llama',
    color='tab:green', hatch='xx', edgecolor='black', zorder=3
)

# --- Formattazione ---
ax.set_ylabel('Error Rate')
ax.set_xlabel('True Label')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 0.9)

ax.grid(axis='y', linestyle='-', alpha=0.5, color='gray', zorder=0)
ax.set_axisbelow(True)

# Legenda con titolo aggiornato
ax.legend(loc='upper left', title='Model', framealpha=1.0, edgecolor='black')

plt.tight_layout()

# --- Salvataggio ---
plt.savefig('Results/Plots/error_rate_analysis.pdf')
plt.show()


=== Error Rates per modello: BERT ===
Classe          | Error Rate | Corrette / Totali
--------------------------------------------------
\textit{l1} (0-39) |   17.101%  | 4038 / 4871
\textit{l2} (40-59) |   64.025%  | 1085 / 3016
\textit{l3} (60-74) |   35.390%  | 2985 / 4620
\textit{l4} (75-99) |   43.047%  | 2871 / 5041
\textit{l5} (100) |   80.809%  |  370 / 1928
--------------------------------------------------

=== Error Rates per modello: RoBERTa ===
Classe          | Error Rate | Corrette / Totali
--------------------------------------------------
\textit{l1} (0-39) |   14.001%  | 4189 / 4871
\textit{l2} (40-59) |   56.200%  | 1321 / 3016
\textit{l3} (60-74) |   32.056%  | 3139 / 4620
\textit{l4} (75-99) |   46.320%  | 2706 / 5041
\textit{l5} (100) |   79.409%  |  397 / 1928
--------------------------------------------------

=== Error Rates per modello: Llama ===
Classe          | Error Rate | Corrette / Totali
--------------------------------------------------
\textit{l1} (

/tmp/ipykernel_249192/987903189.py:129: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()


In [34]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import cycle
from matplotlib.patches import Patch

# ==============================
# CONFIGURAZIONE COSTANTI
# ==============================
TAB_COLORS = [
    "tab:blue", "tab:orange", "tab:green", "tab:red", 
    "tab:purple", "tab:brown", "tab:pink", "tab:gray"
]

# Pattern di default (usati se il modello non ha un pattern specifico)
DEFAULT_HATCHES = [
    "///", "\\\\", "xxx", "|||", "+++", "---", "ooo", "***"
]

# Pattern specifici richiesti
FIXED_HATCHES = {
    "BERT": "//", 
    "RoBERTa": "\\\\", 
    "Llama": "xx",
    "Mistral": "||"
}

def create_model_styles(model_names):
    """
    Crea la mappa colori e hatch basata sui nomi dei modelli.
    Dà priorità ai pattern specifici definiti in FIXED_HATCHES.
    """
    color_cycle = cycle(TAB_COLORS)
    hatch_cycle = cycle(DEFAULT_HATCHES)
    
    color_map = {}
    hatch_map = {}
    
    for model_name in model_names:
        if model_name not in color_map:
            # Assegna colore dal ciclo
            color_map[model_name] = next(color_cycle)
            
            # Assegna hatch: se esiste in FIXED_HATCHES usa quello, altrimenti ciclo
            if model_name in FIXED_HATCHES:
                hatch_map[model_name] = FIXED_HATCHES[model_name]
            else:
                hatch_map[model_name] = next(hatch_cycle)
                
    return color_map, hatch_map

def apply_hatches_only(ax, model_order, hatch_map):
    for i, container in enumerate(ax.containers):
        if i >= len(model_order): break
        model_name = model_order[i]
        hatch = hatch_map.get(model_name, "")
        for bar in container:
            bar.set_hatch(hatch)
            bar.set_edgecolor("black")
            # Usa uno spessore molto piccolo. Se vuoi bordi quasi invisibili metti 0.2
            bar.set_linewidth(0.4) 

def apply_hatches_to_boxplot(ax, model_order, hatch_map, color_map, n_groups):
    artists = [patch for patch in ax.patches if isinstance(patch, plt.matplotlib.patches.PathPatch)]
    for i, artist in enumerate(artists):
        model_idx = i // n_groups
        if model_idx < len(model_order):
            model_name = model_order[model_idx]
            artist.set_hatch(hatch_map.get(model_name, ""))
            artist.set_facecolor(color_map.get(model_name, "gray"))
            artist.set_edgecolor("black")
            artist.set_linewidth(0.6) # Un po' più spesso per le scatole del boxplot

def create_legend_with_patterns(model_order, color_map, hatch_map):
    """
    Crea una legenda personalizzata che mostra sia colore che pattern per ogni modello.
    Utile per utenti daltonici.
    """
    legend_elements = []
    for model_name in model_order:
        color = color_map.get(model_name, "gray")
        hatch = hatch_map.get(model_name, "")
        
        # Crea un patch (rettangolo) con colore, pattern e bordo
        patch = Patch(
            facecolor=color,
            edgecolor='black',
            hatch=hatch,
            label=model_name,
            linewidth=0.5  # BORDO SOTTILE
        )
        legend_elements.append(patch)
    
    return legend_elements

def compare_models_errors(
    dfs, model_names_raw,
    col_orientation="Orientation",
    col_label="label",
    col_pred="predicted_label"
):
    # ==============================
    # MAPPATURA NOMI RICHIESTA
    # ==============================
    NAME_MAPPING = {
        'Bert': 'BERT',
        'RobertaLarge': 'RoBERTa',
        'LLama3': 'Llama'
    }

    # Applichiamo subito la trasformazione ai nomi dei modelli
    model_names = [NAME_MAPPING.get(name, name) for name in model_names_raw]

    print("="*100)
    print("INIZIO ANALISI COMPARATIVA ERRORI TRA MODELLI")
    print("="*100)
    
    # --- 1. VERIFICA INPUT ---
    print("\n" + "─"*100)
    print("FASE 1: VERIFICA E PREPARAZIONE INPUT")
    print("─"*100)
    
    if not isinstance(dfs, list): dfs = [dfs]
    if not isinstance(model_names, list): model_names = [model_names]
    
    print(f"\n✓ Numero di modelli da analizzare: {len(model_names)}")
    print(f"✓ Nomi dei modelli (mappati): {model_names}")
    print(f"\n✓ Colonne utilizzate:")
    print(f"  - Orientamento: '{col_orientation}'")
    print(f"  - Etichetta vera: '{col_label}'")
    print(f"  - Etichetta predetta: '{col_pred}'")
    
    # Mappa stili
    color_map, hatch_map = create_model_styles(model_names)
    print(f"\n✓ Stili assegnati ai modelli:")
    for model in model_names:
        print(f"  - {model}: Colore={color_map[model]}, Texture={hatch_map[model]}")
    
    # --- 2. UNIONE DATI ---
    print("\n" + "─"*100)
    print("FASE 2: ELABORAZIONE DATI PER OGNI MODELLO")
    print("─"*100)
    
    combined_data = []
    
    # Iteriamo usando i nomi mappati
    for idx, (df, name) in enumerate(zip(dfs, model_names)):
        print(f"\n{'='*80}")
        print(f"MODELLO {idx+1}/{len(model_names)}: {name}")
        print(f"{'='*80}")
        
        temp_df = df.copy()
        
        print(f"\n📊 Dimensioni dataset: {temp_df.shape[0]} righe × {temp_df.shape[1]} colonne")
        
        # Sostituisci "Non labelled" con "Non labelled" nel dataframe
        temp_df[col_orientation] = temp_df[col_orientation].replace("Non labelled", "Non labelled")
        
        # Calcolo errore se non esiste
        if "isError" not in temp_df.columns:
            print(f"\n🔧 Calcolo colonna 'isError' (label != predicted_label)...")
            temp_df["isError"] = (temp_df[col_label] != temp_df[col_pred]).astype(int)
        else:
            # Assicuriamoci che isError sia int (0 o 1) per i calcoli successivi
            temp_df["isError"] = temp_df["isError"].astype(int)

        n_errors = temp_df["isError"].sum()
        n_correct = len(temp_df) - n_errors
        print(f"    ✓ Predizioni corrette: {n_correct} ({n_correct/len(temp_df)*100:.1f}%)")
        print(f"    ✓ Predizioni errate: {n_errors} ({n_errors/len(temp_df)*100:.1f}%)")
        
        # ==============================================================================
        # NUOVA FUNZIONALITÀ: CALCOLO ACCURACY PER FOLD
        # ==============================================================================
        # Cerchiamo la colonna fold in modo case-insensitive (es. "fold", "Fold", "FOLD")
        fold_col = next((c for c in temp_df.columns if c.lower() == 'fold'), None)
        
        if fold_col:
            print(f"\n📈 ACCURACY PER FOLD (basato su colonna '{fold_col}'):")
            # Raggruppa per fold. Accuracy = 1 - Error Rate (media di isError)
            fold_stats = temp_df.groupby(fold_col)["isError"].agg(['count', 'mean'])
            fold_stats["accuracy"] = 1 - fold_stats["mean"]
            
            print(f"    {'Fold':<10} | {'Samples':<8} | {'Accuracy':<10}")
            print(f"    {'-'*35}")
            
            accuracies = []
            for fold_val, row in fold_stats.iterrows():
                acc = row['accuracy']
                accuracies.append(acc)
                # Formattiamo il fold (toglie .0 se è float intero)
                fold_str = str(int(fold_val)) if isinstance(fold_val, float) and fold_val.is_integer() else str(fold_val)
                print(f"    {fold_str:<10} | {int(row['count']):<8} | {acc:.2%}")
            
            avg_acc = np.mean(accuracies)
            std_acc = np.std(accuracies)
            print(f"    {'-'*35}")
            print(f"    MEDIA      : {avg_acc:.2%} (± {std_acc:.2%})")
        else:
            print("\n⚠️ Colonna 'Fold' non trovata. Impossibile calcolare accuracy per fold.")
        # ==============================================================================


        # Mostra esempi di dati con errori
        print(f"\n📋 Primi 5 esempi di predizioni (corrette ed errate):")
        sample_cols = [col_orientation, col_label, col_pred, "isError"]
        print(temp_df[sample_cols].head(5).to_string(index=False))
        
        # Calcolo error_distance
        print(f"\n🔧 Calcolo 'error_distance' (|label - predicted|)...")
        temp_df["error_distance"] = abs(temp_df[col_label] - temp_df[col_pred])
        print(f"    Range error_distance: {temp_df['error_distance'].min():.2f} - {temp_df['error_distance'].max():.2f}")
        print(f"    Media error_distance: {temp_df['error_distance'].mean():.3f}")
        
        print(f"\n📋 Distribuzione error_distance:")
        dist_counts = temp_df['error_distance'].value_counts().sort_index()
        total_samples = len(temp_df)
        for dist, count in dist_counts.items():
            percentage = count/total_samples*100
            if dist == 0:
                print(f"    Distanza {int(dist)} (CORRETTI): {count} casi ({percentage:.1f}%)")
            else:
                print(f"    Distanza {int(dist)}: {count} casi ({percentage:.1f}%)")
        
        # Riepilogo corretti vs errati
        n_correct = (temp_df['error_distance'] == 0).sum()
        n_errors = (temp_df['error_distance'] > 0).sum()
        print(f"\n    ✅ RIEPILOGO:")
        print(f"      Esempi CORRETTI: {n_correct} ({n_correct/total_samples*100:.1f}%)")
        print(f"      Esempi ERRATI: {n_errors} ({n_errors/total_samples*100:.1f}%)")
        
        # Calcolo error_direction
        print(f"\n🔧 Calcolo 'error_direction' (predicted - label)...")
        print(f"    (valori positivi = bias verso destra, negativi = bias verso sinistra)")

        temp_df["error_direction"] = temp_df[col_pred] - temp_df[col_label]
        print(f"    Range error_direction: {temp_df['error_direction'].min():.2f} - {temp_df['error_direction'].max():.2f}")
        print(f"    Media error_direction: {temp_df['error_direction'].mean():.3f}")

        # --- ************************************* ---
        print(f"\n🔍 Dettaglio Error Direction (solo casi con errore):")
        
        # Filtriamo solo le righe dove error_distance > 0
        error_directions_only = temp_df.loc[temp_df['error_distance'] > 0, "error_direction"]
        
        print(error_directions_only)

        # OPZIONALE: Se vuoi vedere la distribuzione degli errori (es. quanti sono -1, quanti +2, etc.)
        print(f"\n📊 Distribuzione degli errori:")
        print(error_directions_only.value_counts().sort_index())

        # Analisi per orientamento
        print(f"\n📊 STATISTICHE PER ORIENTAMENTO:")
        print(f"    {'─'*76}")

        temp_df["error_direction"] = temp_df[col_pred] - temp_df[col_label]

        # --- VERIFICA CALCOLO ---
        print(f"\n🔍 Verifica Error Direction (Label vs Pred vs Diff):")
        
        # Selezioniamo solo le colonne che ci interessano
        cols_to_show = [col_label, col_pred, "error_direction"]
        
        # Filtriamo le righe con errore e mostriamo le colonne selezionate
        errors_debug = temp_df.loc[temp_df['error_distance'] > 0, cols_to_show]
        
        print(errors_debug)
        orientations = temp_df[col_orientation].unique()
        for orient in sorted(orientations):
            orient_data = temp_df[temp_df[col_orientation] == orient]
            n_total = len(orient_data)
            n_errors = orient_data["isError"].sum()
            error_rate = n_errors / n_total if n_total > 0 else 0
            mean_dist = orient_data["error_distance"].mean()
            mean_dir = orient_data["error_direction"].mean()
            
            print(f"\n    {orient}:")
            print(f"      • Campioni totali: {n_total}")
            print(f"      • Errori: {n_errors} (Error Rate: {error_rate:.3f})")
            print(f"      • MAE (Mean Absolute Error): {mean_dist:.3f}")
            print(f"      • Bias medio (error_direction): {mean_dir:+.3f}")
        
        # Assegna il nome MAPPATO (es. BERT) alla colonna Model
        temp_df["Model"] = name
        combined_data.append(temp_df)
    
    # --- 3. UNIONE FINALE ---
    print("\n" + "─"*100)
    print("FASE 3: UNIONE DATASET E PREPARAZIONE PER I GRAFICI")
    print("─"*100)
    
    big_df = pd.concat(combined_data, ignore_index=True)
    print(f"\n✓ Dataset combinato creato: {big_df.shape[0]} righe totali")
    print(f"✓ Distribuzione per modello:")
    for model in model_names:
        count = len(big_df[big_df["Model"] == model])
        print(f"  - {model}: {count} righe")
    
    sns.set_theme(style="whitegrid")
    
    # Ordini per il plot
    ORIENTATION_ORDER = [
        "Far Left", "Slightly Left", "Non labelled", "Slightly Right", "Far Right"
    ]
    MODEL_ORDER = model_names
    
    print(f"\n✓ Ordine orientamenti per i grafici: {ORIENTATION_ORDER}")
    print(f"\n✓ Ordine modelli per i grafici: {MODEL_ORDER}")
    
    # Preparazione DataFrame ordinato
    big_df_sorted = big_df.copy()
    big_df_sorted[col_orientation] = pd.Categorical(
        big_df_sorted[col_orientation], categories=ORIENTATION_ORDER, ordered=True
    )
    big_df_sorted["Model"] = pd.Categorical(
        big_df_sorted["Model"], categories=MODEL_ORDER, ordered=True
    )
    big_df_sorted = big_df_sorted.sort_values([col_orientation, "Model"])
    
    # --- TABELLA RIEPILOGATIVA ERROR RATE ---
    print("\n" + "="*100)
    print("TABELLA RIEPILOGATIVA: ERROR RATE PER ORIENTAMENTO E MODELLO")
    print("="*100)
    print("\n(Questi sono i valori che verranno visualizzati nel primo grafico)\n")
    
    error_rate_table = big_df_sorted.groupby([col_orientation, "Model"])["isError"].mean().unstack(fill_value=0)
    print(error_rate_table.to_string())
    
    # --- TABELLA RIEPILOGATIVA MAE ---
    print("\n" + "="*100)
    print("TABELLA RIEPILOGATIVA: MAE (Mean Absolute Error) PER ORIENTAMENTO E MODELLO")
    print("="*100)
    print("\n(Questi sono i valori che verranno visualizzati nel secondo grafico)\n")
    
    mae_table = big_df_sorted.groupby([col_orientation, "Model"])["error_distance"].mean().unstack(fill_value=0)
    print(mae_table.to_string())
    
    # --- TABELLA RIEPILOGATIVA BIAS ---
    print("\n" + "="*100)
    print("TABELLA RIEPILOGATIVA: BIAS DIREZIONALE (solo per errori) PER ORIENTAMENTO E MODELLO")
    print("="*100)
    print("\n(Questi sono i valori che verranno visualizzati nel terzo grafico)")
    print("Nota: Valori positivi = bias verso destra, negativi = bias verso sinistra\n")
    
    errors_only = big_df[big_df["isError"] == 1].copy()
    errors_only[col_orientation] = pd.Categorical(
        errors_only[col_orientation], categories=ORIENTATION_ORDER, ordered=True
    )
    
    print(f"📊 Numero totale di errori considerati: {len(errors_only)}")
    print(f"    Distribuzione per modello:")
    for model in model_names:
        count = len(errors_only[errors_only["Model"] == model])
        print(f"    - {model}: {count} errori")
    
    print("\n")
    bias_table = errors_only.groupby([col_orientation, "Model"])["error_direction"].mean().unstack(fill_value=0)
    print(bias_table.to_string())
    
    # --- STATISTICHE BOXPLOT ---
    print("\n" + "="*100)
    print("STATISTICHE DISTRIBUZIONE BIAS DIREZIONALE (per il boxplot)")
    print("="*100)
    print("\n(Quartili, mediana, outliers per ogni orientamento e modello)\n")
    
    for orient in ORIENTATION_ORDER:
        orient_errors = errors_only[errors_only[col_orientation] == orient]
        if len(orient_errors) > 0:
            print(f"\n{orient}:")
            print(f"  {'─'*70}")
            for model in MODEL_ORDER:
                model_data = orient_errors[orient_errors["Model"] == model]["error_direction"]
                if len(model_data) > 0:
                    q1 = model_data.quantile(0.25)
                    median = model_data.median()
                    q3 = model_data.quantile(0.75)
                    min_val = model_data.min()
                    max_val = model_data.max()
                    mean_val = model_data.mean()
                    
                    print(f"\n  {model} (n={len(model_data)}):")
                    print(f"    • Min: {min_val:+.2f}")
                    print(f"    • Q1 (25%): {q1:+.2f}")
                    print(f"    • Mediana (50%): {median:+.2f}")
                    print(f"    • Media: {mean_val:+.2f}")
                    print(f"    • Q3 (75%): {q3:+.2f}")
                    print(f"    • Max: {max_val:+.2f}")
                    print(f"    • IQR: {q3-q1:.2f}")
                else:
                    print(f"\n  {model}: Nessun errore")
    
    # Crea legenda personalizzata con pattern
    legend_elements = create_legend_with_patterns(MODEL_ORDER, color_map, hatch_map)
    
    print("\n" + "="*100)
    print("ANALISI COMPLETATA CON SUCCESSO!")
    print("="*100)
    print(f"\n📊 Dati preparati per plotting")
    print(f"    Usa il dizionario 'results' restituito per creare i grafici nelle celle successive")
    
    # RETURN MODIFICATO: restituisce un dizionario con tutti i dati necessari per il plotting
    return {
        'color_map': color_map,
        'hatch_map': hatch_map,
        'big_df_sorted': big_df_sorted,
        'errors_only': errors_only,
        'legend_elements': legend_elements,
        'MODEL_ORDER': MODEL_ORDER,
        'ORIENTATION_ORDER': ORIENTATION_ORDER,
        'col_orientation': col_orientation
    }

# CHIAMATA MODIFICATA: salva il risultato in una variabile
results = compare_models_errors(
    [bertErrors, robertaErrors, llamaErrors], 
    ["Bert", "RobertaLarge", "LLama3"]
)

INIZIO ANALISI COMPARATIVA ERRORI TRA MODELLI

────────────────────────────────────────────────────────────────────────────────────────────────────
FASE 1: VERIFICA E PREPARAZIONE INPUT
────────────────────────────────────────────────────────────────────────────────────────────────────

✓ Numero di modelli da analizzare: 3
✓ Nomi dei modelli (mappati): ['BERT', 'RoBERTa', 'Llama']

✓ Colonne utilizzate:
  - Orientamento: 'Orientation'
  - Etichetta vera: 'label'
  - Etichetta predetta: 'predicted_label'

✓ Stili assegnati ai modelli:
  - BERT: Colore=tab:blue, Texture=//
  - RoBERTa: Colore=tab:orange, Texture=\\
  - Llama: Colore=tab:green, Texture=xx

────────────────────────────────────────────────────────────────────────────────────────────────────
FASE 2: ELABORAZIONE DATI PER OGNI MODELLO
────────────────────────────────────────────────────────────────────────────────────────────────────

MODELLO 1/3: BERT

📊 Dimensioni dataset: 19476 righe × 16 colonne
    ✓ Predizioni corrette:

/tmp/ipykernel_249192/1101851082.py:328: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  error_rate_table = big_df_sorted.groupby([col_orientation, "Model"])["isError"].mean().unstack(fill_value=0)
/tmp/ipykernel_249192/1101851082.py:337: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  mae_table = big_df_sorted.groupby([col_orientation, "Model"])["error_distance"].mean().unstack(fill_value=0)
/tmp/ipykernel_249192/1101851082.py:359: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True t

In [35]:
results = compare_models_errors(
    [bertErrors, robertaErrors, llamaErrors, mistralErrors], 
    ["Bert", "RobertaLarge", "LLama3", "Mistral"]
)

INIZIO ANALISI COMPARATIVA ERRORI TRA MODELLI

────────────────────────────────────────────────────────────────────────────────────────────────────
FASE 1: VERIFICA E PREPARAZIONE INPUT
────────────────────────────────────────────────────────────────────────────────────────────────────

✓ Numero di modelli da analizzare: 4
✓ Nomi dei modelli (mappati): ['BERT', 'RoBERTa', 'Llama', 'Mistral']

✓ Colonne utilizzate:
  - Orientamento: 'Orientation'
  - Etichetta vera: 'label'
  - Etichetta predetta: 'predicted_label'

✓ Stili assegnati ai modelli:
  - BERT: Colore=tab:blue, Texture=//
  - RoBERTa: Colore=tab:orange, Texture=\\
  - Llama: Colore=tab:green, Texture=xx
  - Mistral: Colore=tab:red, Texture=||

────────────────────────────────────────────────────────────────────────────────────────────────────
FASE 2: ELABORAZIONE DATI PER OGNI MODELLO
────────────────────────────────────────────────────────────────────────────────────────────────────

MODELLO 1/4: BERT

📊 Dimensioni dataset: 

/tmp/ipykernel_249192/1101851082.py:328: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  error_rate_table = big_df_sorted.groupby([col_orientation, "Model"])["isError"].mean().unstack(fill_value=0)
/tmp/ipykernel_249192/1101851082.py:337: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  mae_table = big_df_sorted.groupby([col_orientation, "Model"])["error_distance"].mean().unstack(fill_value=0)
/tmp/ipykernel_249192/1101851082.py:359: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True t

In [36]:
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
from matplotlib.patches import Patch

matplotlib.use("pgf")
plt.rcParams.update({
    "pgf.texsystem": "pdflatex",
    "text.usetex": True,
    "font.family": "serif",
    "axes.labelsize": 14,
    "font.size": 14,
    "legend.fontsize": 11,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
})

color_map       = results['color_map']
hatch_map       = results['hatch_map']
errors_only     = results['errors_only']
MODEL_ORDER     = results['MODEL_ORDER']
ORIENTATION_ORDER = results['ORIENTATION_ORDER']
col_orientation = results['col_orientation']

# Calcola bias medio per orientamento x modello (solo errori)
bias_table = (
    errors_only
    .groupby([col_orientation, "Model"])["error_direction"]
    .mean()
    .unstack(fill_value=0)
    .reindex(index=ORIENTATION_ORDER, columns=MODEL_ORDER, fill_value=0)
)

orientations = ORIENTATION_ORDER
x = np.arange(len(orientations))
n_models = len(MODEL_ORDER)
width = 0.18

fig, ax = plt.subplots(figsize=(12, 6))

offsets = np.linspace(-(n_models - 1) / 2, (n_models - 1) / 2, n_models) * width

for i, model in enumerate(MODEL_ORDER):
    vals = [bias_table.loc[o, model] if o in bias_table.index else 0.0 for o in orientations]
    ax.bar(
        x + offsets[i], vals, width,
        color=color_map[model],
        hatch=hatch_map[model],
        edgecolor='black',
        linewidth=0.5,
        zorder=3,
        label=model
    )

ax.axhline(0, color='black', linewidth=0.8, zorder=4)

# x labels — sostituisci "Non labelled" con testo su due righe
xtick_labels = [o.replace("Non labelled", "Non labelled\nLeaning") for o in orientations]
ax.set_xticks(x)
ax.set_xticklabels(xtick_labels)

ax.set_ylabel(r'Mean Error Direction $(\hat{y} - y)$')
ax.set_ylim(-1.1, 1.2)
ax.set_yticks(np.arange(-1.0, 1.25, 0.25))
ax.grid(axis='y', linestyle='-', alpha=0.5, color='gray', zorder=0)
ax.set_axisbelow(True)

legend_elements = create_legend_with_patterns(MODEL_ORDER, color_map, hatch_map)
ax.legend(handles=legend_elements, title='Model', loc='upper left', framealpha=1.0, edgecolor='black')

plt.tight_layout()
plt.savefig('Results/Plots/bias_direction_by_orientation.pdf')
plt.show()

/tmp/ipykernel_249192/4080026251.py:28: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby([col_orientation, "Model"])["error_direction"]
/tmp/ipykernel_249192/4080026251.py:73: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()


In [37]:
# PLOT ERROR RATE
print("\n" + "─"*100)
print("FASE 4: GENERAZIONE GRAFICI + OUTPUT TESTUALE")
print("─"*100)

# Estrai i dati
big_df_sorted = results['big_df_sorted']
color_map = results['color_map']
hatch_map = results['hatch_map']
legend_elements = results['legend_elements']
MODEL_ORDER = results['MODEL_ORDER']
ORIENTATION_ORDER = results['ORIENTATION_ORDER']
col_orientation = results['col_orientation']

# Stile
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "font.size": 14,
    "axes.labelsize": 14,
    "legend.fontsize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "axes.linewidth": 1.5,
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
})

plt.figure(figsize=(12, 6))
ax = sns.barplot(
    data=big_df_sorted,
    x=col_orientation,
    y="isError",
    hue="Model",
    hue_order=MODEL_ORDER,
    order=ORIENTATION_ORDER,
    palette=color_map,
    errorbar=None,
    edgecolor="black",
)

for i, container in enumerate(ax.containers):
    if i >= len(MODEL_ORDER): break
    model_name = MODEL_ORDER[i]
    hatch = hatch_map.get(model_name, "")
    for bar in container:
        bar.set_hatch(hatch)
        bar.set_edgecolor("black")
        bar.set_linewidth(1.2)

for spine in ax.spines.values():
    spine.set_linewidth(1.5)

plt.ylabel("Error Rate (Mean)")
plt.xlabel("Leaning")
plt.legend(handles=legend_elements, title="Model", loc="upper center")
plt.tight_layout()
plt.savefig('Results/Plots/politicalErrorRates.pdf', bbox_inches='tight')
plt.show()

# ------------------------------------------------------------------
# OUTPUT TESTUALE
# ------------------------------------------------------------------
import pandas as pd

# 1. Calcola media, std e N per ogni combinazione
summary = big_df_sorted.groupby([col_orientation, "Model"])["isError"].agg(
    ErrorRate_Mean="mean",
    ErrorRate_Std="std",
    N="count"
).reset_index()

# 2. Ordina come nel grafico
summary[col_orientation] = pd.Categorical(summary[col_orientation], categories=ORIENTATION_ORDER, ordered=True)
summary["Model"] = pd.Categorical(summary["Model"], categories=MODEL_ORDER, ordered=True)
summary = summary.sort_values([col_orientation, "Model"])

# 3. Stampa a console
print("\n" + "="*100)
print("RISULTATI TESTUALI - Error Rate per Orientamento Politico")
print("="*100)
print(summary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# 4. Tabella pivot per lettura rapida
pivot_mean = summary.pivot(index=col_orientation, columns="Model", values="ErrorRate_Mean")
print("\n--- Tabella riassuntiva (Mean Error Rate) ---")
print(pivot_mean.to_string(float_format=lambda x: f"{x:.3f}"))

# 5. Salva su file
os.makedirs("Results/Tables", exist_ok=True)
summary.to_csv("Results/Tables/political_error_rates.csv", index=False)
pivot_mean.to_csv("Results/Tables/political_error_rates_pivot.csv")

# 6. Esporta in LaTeX per il paper
latex_table = pivot_mean.to_latex(
    float_format="%.3f",
    caption="Mean error rate by political leaning and model.",
    label="tab:political_error",
    na_rep="-"
)
with open("Results/Tables/political_error_rates.tex", "w") as f:
    f.write(latex_table)

print("\nFile salvati:")
print(" - Results/Tables/political_error_rates.csv")
print(" - Results/Tables/political_error_rates_pivot.csv")
print(" - Results/Tables/political_error_rates.tex")


────────────────────────────────────────────────────────────────────────────────────────────────────
FASE 4: GENERAZIONE GRAFICI + OUTPUT TESTUALE
────────────────────────────────────────────────────────────────────────────────────────────────────

RISULTATI TESTUALI - Error Rate per Orientamento Politico
   Orientation   Model  ErrorRate_Mean  ErrorRate_Std    N
      Far Left    BERT          0.7054         0.4561 1015
      Far Left RoBERTa          0.7163         0.4510 1015
      Far Left   Llama          0.7192         0.4496 1015
      Far Left Mistral          0.6355         0.4815 1015
 Slightly Left    BERT          0.5771         0.4941 2407
 Slightly Left RoBERTa          0.5152         0.4999 2407
 Slightly Left   Llama          0.4728         0.4994 2407
 Slightly Left Mistral          0.5663         0.4957 2407
  Non labelled    BERT          0.3317         0.4709 6422
  Non labelled RoBERTa          0.3186         0.4660 6422
  Non labelled   Llama          0.3242     

/tmp/ipykernel_249192/1176094537.py:60: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_249192/1176094537.py:68: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary = big_df_sorted.groupby([col_orientation, "Model"])["isError"].agg(


In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ast
import matplotlib

matplotlib.use("pgf")
plt.rcParams.update({
    "pgf.texsystem": "pdflatex",
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": [],
    "axes.labelsize": 14,
    "font.size": 14,
    "legend.fontsize": 12,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "pgf.rcfonts": False,
})

labelMapping = {
    0: r"\textit{l1} (0-39)",
    1: r"\textit{l2} (40-59)",
    2: r"\textit{l3} (60-74)",
    3: r"\textit{l4} (75-99)",
    4: r"\textit{l5} (100)"
}

def calculate_mean_error_direction(df, model_name):
    total_cm = np.zeros((5, 5), dtype=int)
    for matrix_str in df['Confusion_Matrix']:
        cm = np.array(ast.literal_eval(matrix_str))
        total_cm += cm

    biases = []
    total_samples_per_class = []

    for t in range(5):
        total_samples = total_cm[t, :].sum()
        total_samples_per_class.append(total_samples)

        weighted_error_sum = 0
        for p in range(5):
            count = total_cm[t, p]
            diff = p - t
            weighted_error_sum += count * diff

        mean_bias = weighted_error_sum / total_samples if total_samples > 0 else 0
        biases.append(mean_bias)

    print(f"\n=== Mean Error Direction per modello: {model_name} ===")
    print("Classe          |  Mean Bias  |  Corrette / Totali  |  Interpretazione")
    print("-" * 70)
    for i, (bias, total) in enumerate(zip(biases, total_samples_per_class)):
        label = labelMapping[i]
        correct = total_cm[i, i] if total > 0 else 0
        if total > 0:
            interp = "→ sovrastima" if bias > 0.15 else \
                     "→ sottostima" if bias < -0.15 else \
                     "≈ neutro"
            print(f"{label:<15} | {bias:>10.3f}  | {correct:>4d} / {total:>4d}      | {interp}")
        else:
            print(f"{label:<15} |       N/A      |    0 / 0             | (nessun esempio)")
    print("-" * 70)

    return biases

# --- CALCOLO ---
bert_bias    = calculate_mean_error_direction(bertResults,    "BERT")
roberta_bias = calculate_mean_error_direction(robertaResults, "RoBERTa")
llama_bias   = calculate_mean_error_direction(llamaResults,   "Llama")
mistral_bias = calculate_mean_error_direction(mistralResults, "Mistral")

# --- PLOTTING ---
labels = [labelMapping[i] for i in range(5)]
x = np.arange(len(labels))
width = 0.19  # ridotto per 4 modelli

fig, ax = plt.subplots(figsize=(13, 6))

ax.axhline(0, color='black', linewidth=1.2, zorder=2)

ax.bar(x - 1.5*width, bert_bias,    width, label='BERT',    color='tab:blue',   hatch='//',   edgecolor='black', zorder=3)
ax.bar(x - 0.5*width, roberta_bias, width, label='RoBERTa', color='tab:orange', hatch='\\\\', edgecolor='black', zorder=3)
ax.bar(x + 0.5*width, llama_bias,   width, label='Llama',   color='tab:green',  hatch='xx',   edgecolor='black', zorder=3)
ax.bar(x + 1.5*width, mistral_bias, width, label='Mistral', color='tab:red',    hatch='||',   edgecolor='black', zorder=3)

ax.set_ylabel(r'Mean Error Direction $(\hat{y} - y)$')
ax.set_xlabel('True Label (Reliability)')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.grid(axis='y', linestyle='-', alpha=0.3, color='gray', zorder=0)
ax.set_axisbelow(True)
ax.legend(loc='upper right', title='Model', framealpha=1.0, edgecolor='black')

plt.tight_layout()
plt.savefig('Results/Plots/mean_error_direction.pdf')


=== Mean Error Direction per modello: BERT ===
Classe          |  Mean Bias  |  Corrette / Totali  |  Interpretazione
----------------------------------------------------------------------
\textit{l1} (0-39) |      0.393  | 4038 / 4871      | → sovrastima
\textit{l2} (40-59) |      0.980  | 1085 / 3016      | → sovrastima
\textit{l3} (60-74) |      0.123  | 2985 / 4620      | ≈ neutro
\textit{l4} (75-99) |     -0.356  | 2871 / 5041      | → sottostima
\textit{l5} (100) |     -1.291  |  370 / 1928      | → sottostima
----------------------------------------------------------------------

=== Mean Error Direction per modello: RoBERTa ===
Classe          |  Mean Bias  |  Corrette / Totali  |  Interpretazione
----------------------------------------------------------------------
\textit{l1} (0-39) |      0.312  | 4189 / 4871      | → sovrastima
\textit{l2} (40-59) |      0.795  | 1321 / 3016      | → sovrastima
\textit{l3} (60-74) |      0.032  | 3139 / 4620      | ≈ neutro
\textit{l4} (7

In [39]:
results = compare_models_errors(
    [bertErrors, robertaErrors, llamaErrors, mistralErrors], 
    ["Bert", "RobertaLarge", "LLama3", "Mistral"]
)

INIZIO ANALISI COMPARATIVA ERRORI TRA MODELLI

────────────────────────────────────────────────────────────────────────────────────────────────────
FASE 1: VERIFICA E PREPARAZIONE INPUT
────────────────────────────────────────────────────────────────────────────────────────────────────

✓ Numero di modelli da analizzare: 4
✓ Nomi dei modelli (mappati): ['BERT', 'RoBERTa', 'Llama', 'Mistral']

✓ Colonne utilizzate:
  - Orientamento: 'Orientation'
  - Etichetta vera: 'label'
  - Etichetta predetta: 'predicted_label'

✓ Stili assegnati ai modelli:
  - BERT: Colore=tab:blue, Texture=//
  - RoBERTa: Colore=tab:orange, Texture=\\
  - Llama: Colore=tab:green, Texture=xx
  - Mistral: Colore=tab:red, Texture=||

────────────────────────────────────────────────────────────────────────────────────────────────────
FASE 2: ELABORAZIONE DATI PER OGNI MODELLO
────────────────────────────────────────────────────────────────────────────────────────────────────

MODELLO 1/4: BERT

📊 Dimensioni dataset: 

/tmp/ipykernel_249192/1101851082.py:328: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  error_rate_table = big_df_sorted.groupby([col_orientation, "Model"])["isError"].mean().unstack(fill_value=0)
/tmp/ipykernel_249192/1101851082.py:337: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  mae_table = big_df_sorted.groupby([col_orientation, "Model"])["error_distance"].mean().unstack(fill_value=0)
/tmp/ipykernel_249192/1101851082.py:359: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True t

In [40]:
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import pandas as pd
import os
from matplotlib.patches import Patch

matplotlib.use("pgf")
plt.rcParams.update({
    "pgf.texsystem": "pdflatex",
    "text.usetex": True,
    "font.family": "serif",
    "axes.labelsize": 14,
    "font.size": 14,
    "legend.fontsize": 11,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
})

color_map = results['color_map']
# --- PATTERN FISSI PER IL PDF ---
FIXED_HATCHES = {
    "BERT": "//",
    "RoBERTa": "\\\\",
    "Llama": "xx",
    "Mistral": "||"
}
hatch_map = FIXED_HATCHES # <--- sovrascrive quello da results

errors_only = results['errors_only']
MODEL_ORDER = results['MODEL_ORDER']
ORIENTATION_ORDER = results['ORIENTATION_ORDER']
col_orientation = results['col_orientation']

# Calcola bias medio per orientamento x modello (solo errori)
bias_table = (
    errors_only
  .groupby([col_orientation, "Model"])["error_direction"]
  .mean()
  .unstack(fill_value=0)
  .reindex(index=ORIENTATION_ORDER, columns=MODEL_ORDER, fill_value=0)
)

orientations = ORIENTATION_ORDER
x = np.arange(len(orientations))
n_models = len(MODEL_ORDER)
width = 0.18

fig, ax = plt.subplots(figsize=(12, 6))

offsets = np.linspace(-(n_models - 1) / 2, (n_models - 1) / 2, n_models) * width

for i, model in enumerate(MODEL_ORDER):
    vals = [bias_table.loc[o, model] if o in bias_table.index else 0.0 for o in orientations]
    ax.bar(
        x + offsets[i], vals, width,
        color=color_map[model],
        hatch=hatch_map[model], # usa il pattern fisso
        edgecolor='black',
        linewidth=0.5,
        zorder=3,
        label=model
    )

ax.axhline(0, color='black', linewidth=0.8, zorder=4)

xtick_labels = [o.replace("Non labelled", "Non labelled\nLeaning") for o in orientations]
ax.set_xticks(x)
ax.set_xticklabels(xtick_labels)

ax.set_ylabel(r'Mean Error Direction $(\hat{y} - y)$')
ax.set_ylim(-1.1, 1.2)
ax.set_yticks(np.arange(-1.0, 1.25, 0.25))
ax.grid(axis='y', linestyle='-', alpha=0.5, color='gray', zorder=0)
ax.set_axisbelow(True)

# la legenda userà gli stessi pattern fissi
legend_elements = create_legend_with_patterns(MODEL_ORDER, color_map, hatch_map)
ax.legend(handles=legend_elements, title='Model', loc='upper left', framealpha=1.0, edgecolor='black')

plt.tight_layout()
plt.savefig('Results/Plots/politicalErrorsDirection.pdf')
plt.show()

# ------------------------------------------------------------------
# OUTPUT TESTUALE
# ------------------------------------------------------------------
print("\n" + "="*100)
print("RISULTATI TESTUALI - Mean Error Direction per Orientamento")
print("="*100)

print(bias_table.to_string(float_format=lambda x: f"{x:+.3f}"))

stats_table = (
    errors_only
  .groupby([col_orientation, "Model"])["error_direction"]
  .agg(["mean","std","count"])
  .unstack("Model")
)
print("\n--- Dettaglio con std e numerosità ---")
print(stats_table.to_string(float_format=lambda x: f"{x:.3f}"))

overall_bias = errors_only.groupby("Model")["error_direction"].mean().reindex(MODEL_ORDER)
print("\n--- Bias complessivo per modello ---")
for m, v in overall_bias.items():
    direction = "sovrastima (destra)" if v > 0 else "sottostima (sinistra)"
    print(f"{m:<10}: {v:+.3f} → {direction}")

# Salvataggi (opzionali)
os.makedirs("Results/Tables", exist_ok=True)
bias_table.to_csv("Results/Tables/political_error_direction.csv")
stats_table.to_csv("Results/Tables/political_error_direction_stats.csv")

latex = bias_table.to_latex(
    float_format=lambda x: f"{x:+.3f}",
    caption="Mean error direction $(\\hat{y}-y)$ by political leaning. Positive values indicate overestimation towards the right.",
    label="tab:political_direction",
    escape=False
)
with open("Results/Tables/political_error_direction.tex", "w") as f:
    f.write(latex)

print("\nFile salvati:")
print(" - Results/Tables/political_error_direction.csv")
print(" - Results/Tables/political_error_direction_stats.csv")
print(" - Results/Tables/political_error_direction.tex")

/tmp/ipykernel_249192/444032424.py:38: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby([col_orientation, "Model"])["error_direction"]



RISULTATI TESTUALI - Mean Error Direction per Orientamento
Model            BERT  RoBERTa  Llama  Mistral
Orientation                                   
Far Left       +0.464   +0.538 +0.629   +0.771
Slightly Left  -0.819   -0.731 -0.599   -0.537
Non labelled   +0.097   -0.026 +0.264   +0.069
Slightly Right +0.116   -0.308 -0.081   -0.392
Far Right      +1.001   +0.528 +0.517   +0.332

--- Dettaglio con std e numerosità ---
                 mean                          std                       count                      
Model            BERT  Llama Mistral RoBERTa  BERT Llama Mistral RoBERTa  BERT Llama Mistral RoBERTa
Orientation                                                                                         
Far Left        0.464  0.629   0.771   0.538 2.011 2.056   1.893   2.024   716   730     645     727
Slightly Left  -0.819 -0.599  -0.537  -0.731 1.378 1.533   1.611   1.529  1389  1138    1363    1240
Non labelled    0.097  0.264   0.069  -0.026 1.753 1.695   1.667  

/tmp/ipykernel_249192/444032424.py:83: UserWarning: FigureCanvasPgf is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_249192/444032424.py:96: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby([col_orientation, "Model"])["error_direction"]
